# Classification automatique des réclamations clients

**Pipeline** : EDA -> nettoyage -> déduplication -> split stratifié (catégorie x langue) -> **Logistic Regression (TF-IDF, recherche d'hyperparamétres conjointe)** -> **DistilBERT multilingue (fine-tuning)** -> évaluation comparative (globale + par langue) -> analyse des erreurs (dont erreurs les plus confiantes).

Notebook pensé pour **Google Colab** (GPU recommandé : `Exécution > Modifier le type d'exécution > T4 GPU`).

> Ce notebook fusionne deux pipelines précédents : la rigueur méthodologique du pipeline complet (CV de stabilité, tests de significativité, IC bootstrap, audit qualité) est conservée, et on y ajoute une recherche d'hyperparamétres conjointe vectorizer+classifieur ainsi qu'une étape d'interprétabilité et une analyse des erreurs les plus confiantes.


## 1. Installation des dépendances

In [ ]:
# transformers/datasets/accelerate ne sont pas preinstalles sur Colab ; torch et scikit-learn le sont deja.
# Versions EPINGLEES explicitement (reproductibilite) : deux executions a des dates differentes
# installaient auparavant les dernieres versions disponibles, ce qui peut faire varier les resultats
# (notamment le fine-tuning DistilBERT). Ajuster ces epingles si une version plus recente est requise,
# mais le faire consciemment plutot que de laisser pip resoudre silencieusement.
!pip install -q "langdetect==1.0.9" "statsmodels==0.14.4" "transformers==4.46.3" "datasets==3.1.0" "accelerate==1.1.1"


## 2. Setup : imports, seed, configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import logging
import os
import random
import re
import time
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    classification_report, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay, balanced_accuracy_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import loguniform

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-8s | %(message)s")
logger = logging.getLogger("reclamations")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110


In [ ]:
!pip uninstall -y torchvision


In [ ]:
# --- Configuration centralisee (toutes les constantes du pipeline ici) ---

@dataclass(frozen=True)
class Config:
    seed: int = 42

    raw_path: Path = Path("/content/reclamations_categorie_finale_v3.csv")
    interim_dir: Path = Path("/content/data/interim")
    processed_dir: Path = Path("/content/data/processed")
    models_dir: Path = Path("/content/models")
    reports_dir: Path = Path("/content/reports")

    text_col: str = "texte"
    target_col: str = "categorie_finale"
    excluded_cols: tuple = ("ville", "montant", "montant_num")

    test_size: float = 0.15
    val_size: float = 0.15
    min_stratum_size_warning: int = 30

    near_dup_similarity_threshold: float = 0.92
    # Si un couple quasi-duplique n'a PAS la meme categorie, on ne le supprime jamais
    # automatiquement (cf. section 6) : soit un vrai cas ambigu entre classes proches,
    # soit un faux positif de similarite textuelle -> traite manuellement.
    near_dup_require_same_category: bool = True

    # Nombre d'iterations tirees aleatoirement dans l'espace de recherche
    # conjoint vectorizer + classifieur (section 10).
    baseline_search_n_iter: int = 40

    # La CV a 5 plis sur DistilBERT (section 11.1) entraine 5 modeles complets EN PLUS
    # du modele final : tres couteux sur un GPU Colab gratuit / quota limite. Mettre a
    # False pour sauter directement a l'entrainement final si les ressources sont contraintes.
    run_bert_cv_diagnostic: bool = True

    # Determinisme total (torch inclus). Peut legerement ralentir l'entrainement GPU
    # (cudnn non-deterministe desactive) mais garantit des runs reproductibles.
    deterministic: bool = True

CFG = Config()
for d in (CFG.interim_dir, CFG.processed_dir, CFG.models_dir, CFG.reports_dir, CFG.reports_dir / "figures"):
    d.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int, deterministic: bool = True) -> None:
    """Fixe TOUTES les sources d'alea (python, numpy, torch CPU/GPU) -- pas seulement
    numpy/random comme avant. Sans ca, deux executions avec le meme seed peuvent quand
    meme diverger a cause de l'initialisation des poids torch ou du dropout. A rappeler
    une seconde fois une fois torch importe (cf. section 11), car ici torch n'est pas
    encore installe (section 1 s'execute apres)."""
    import os as _os
    _os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        if deterministic:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except ImportError:
        pass


seed_everything(CFG.seed, CFG.deterministic)
logger.info("Configuration chargee. Seed=%s | run_bert_cv_diagnostic=%s", CFG.seed, CFG.run_bert_cv_diagnostic)


### 2.1 Traçabilité : versions des librairies installées

Les épingles de la section 1 fixent une *cible*, mais l'environnement réellement utilisé (Colab préinstalle certains paquets, une résolution de dépendances peut différer) peut diverger légèrement. On sauvegarde donc ici les versions **effectivement chargées** dans un fichier de traçabilité, pour pouvoir reproduire l'environnement exact d'un run donné.

In [ ]:
# --- Tracabilite : sauvegarde des versions effectivement installees pour ce run ---
import importlib.metadata as _metadata

_TRACKED_PACKAGES = [
    "numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "seaborn",
    "langdetect", "statsmodels", "transformers", "datasets", "accelerate",
    "torch", "spacy", "joblib",
]

def _get_version(pkg: str) -> str:
    """Version installee d'un paquet, ou 'non installe' s'il n'est pas encore disponible
    a ce stade du notebook (ex. spacy, installe plus tard, section 16)."""
    try:
        return _metadata.version(pkg)
    except _metadata.PackageNotFoundError:
        return "non installe"

versions_used = {pkg: _get_version(pkg) for pkg in _TRACKED_PACKAGES}
versions_path = CFG.reports_dir / "versions_librairies.json"
with open(versions_path, "w", encoding="utf-8") as _f:
    json.dump(versions_used, _f, indent=2, ensure_ascii=False)

logger.info("Versions des librairies sauvegardees dans %s", versions_path)
for _pkg, _v in versions_used.items():
    print(f"  {_pkg:15s} {_v}")


## 3. Chargement des données

Deux options : uploader le CSV directement (widget ci-dessous), ou le lire depuis Google Drive (décommenter la cellule suivante).

In [ ]:
from google.colab import files

CFG.raw_path.parent.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()  # selectionnez votre fichier .csv (colonnes attendues : texte, categorie_finale)
uploaded_name = next(iter(uploaded))
Path(uploaded_name).rename(CFG.raw_path)
print(f"Fichier enregistre : {CFG.raw_path}")


In [ ]:
df = pd.read_csv(CFG.raw_path)
logger.info("Donnees chargees : %d lignes, %d colonnes", *df.shape)
df.head(3)


### 3.1 Validation du schéma des données

On vérifie explicitement, juste après le chargement, que les colonnes attendues sont présentes et qu'aucune ligne n'a un texte ou une catégorie vide. Sans ce garde-fou, un CSV légèrement différent (colonne renommée, ligne corrompue) ferait planter le pipeline beaucoup plus loin (souvent au TF-IDF ou au tokenizer) avec un message d'erreur peu clair.

In [ ]:
# --- Validation du schema attendu (evite un plantage tardif avec une erreur peu claire) ---
REQUIRED_COLUMNS = [CFG.text_col, CFG.target_col]


class SchemaValidationError(ValueError):
    """Erreur levee quand le CSV charge ne respecte pas le schema minimal attendu."""


missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_cols:
    raise SchemaValidationError(
        f"Colonne(s) obligatoire(s) absente(s) du CSV : {missing_cols}. "
        f"Colonnes trouvees : {list(df.columns)}. "
        f"Attendu au minimum : {REQUIRED_COLUMNS}."
    )

n_before = len(df)

# Lignes avec texte ou categorie vide/NaN (apres strip) : inutilisables pour le pipeline.
empty_text = df[CFG.text_col].isna() | (df[CFG.text_col].astype(str).str.strip() == "")
empty_target = df[CFG.target_col].isna() | (df[CFG.target_col].astype(str).str.strip() == "")
n_empty_text, n_empty_target = int(empty_text.sum()), int(empty_target.sum())

if n_empty_text or n_empty_target:
    logger.warning(
        "Lignes invalides detectees : %d avec '%s' vide, %d avec '%s' vide -- suppression avant la suite du pipeline.",
        n_empty_text, CFG.text_col, n_empty_target, CFG.target_col,
    )
    invalid_path = CFG.reports_dir / "lignes_invalides_au_chargement.csv"
    df.loc[empty_text | empty_target].to_csv(invalid_path, index=False)
    logger.info("Lignes invalides exportees dans %s (pour audit).", invalid_path)

df = df.loc[~(empty_text | empty_target)].reset_index(drop=True)

logger.info(
    "Schema valide. %d lignes conservees / %d chargees (%d supprimees).",
    len(df), n_before, n_before - len(df),
)
assert len(df) > 0, "Aucune ligne valide apres validation du schema -- verifier le CSV source."


## 4. EDA rapide

In [ ]:
print("Dimensions :", df.shape)
print()
print("Valeurs manquantes (%) :")
print((df.isnull().mean() * 100).round(2))
print()
print(f"Doublons exacts (texte) : {df[CFG.text_col].duplicated().sum()}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
order = df[CFG.target_col].value_counts().index
sns.countplot(data=df, y=CFG.target_col, order=order, ax=ax)
ax.set_title("Distribution des categories de reclamation")
ax.set_xlabel("Nombre de reclamations")
ax.set_ylabel("")
for i, cat in enumerate(order):
    n = (df[CFG.target_col] == cat).sum()
    ax.text(n + 5, i, f"{n} ({n/len(df)*100:.1f}%)", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(CFG.reports_dir / "figures" / "distribution_categories.png", bbox_inches="tight")
plt.show()

ratio_desequilibre = df[CFG.target_col].value_counts().max() / df[CFG.target_col].value_counts().min()
print(f"Ratio de desequilibre (classe majoritaire / minoritaire) : {ratio_desequilibre:.1f}x")


### 4.1 Détection de la langue

In [ ]:
from langdetect import DetectorFactory, detect

DetectorFactory.seed = CFG.seed  # rend langdetect deterministe

def detect_lang_safe(text: str) -> str:
    """Detecte la langue d'un texte, avec repli robuste sur cas degeneres."""
    if not isinstance(text, str) or len(text.strip()) < 3:
        return "unknown"
    try:
        lang = detect(text)
    except Exception:
        return "unknown"
    return lang if lang in ("fr", "en") else "other"

df["langue"] = df[CFG.text_col].apply(detect_lang_safe)
print(df["langue"].value_counts())
print(f"\n% anglais : {(df['langue']=='en').mean()*100:.1f}%")


### 4.1bis Reintegration des corrections de l'audit langue (avant le split)

Meme principe : les corrections de `audit_langdetect_a_completer.csv` (section 13) sont appliquees ICI, **avant** le split stratifie (section 7) et avant la vérification de fuite (section 4.2) -- une correction appliquee apres le split ne changerait pas la composition train/val/test, et une correction appliquee apres l'affichage crosstab ci-dessous fausserait la lecture de la repartition categorie x langue.


In [ ]:
# --- Reintegration des corrections d'un audit langdetect manuel precedent ---
_lang_audit_path = CFG.reports_dir / "audit_langdetect_a_completer.csv"

if _lang_audit_path.exists():
    _audit = pd.read_csv(_lang_audit_path)
    _audit = _audit[_audit["langue_correcte"].notna() & (_audit["langue_correcte"].astype(str).str.strip() != "")]
    if len(_audit):
        # On ne peut recoller que sur le texte tronque sauvegarde dans l'audit (texte_court) ;
        # on matche donc sur les 150 premiers caracteres du texte original, comme lors de la
        # generation de l'audit (cf. sample_for_langdetect_audit, section 13).
        _lookup = dict(zip(_audit["texte_court"], _audit["langue_correcte"]))
        _mask = df[CFG.text_col].str.slice(0, 150).isin(_lookup)
        n_corrected = int(_mask.sum())
        df.loc[_mask, "langue"] = df.loc[_mask, CFG.text_col].str.slice(0, 150).map(_lookup)
        logger.info("%d langues corrigees depuis l'audit manuel (%s)", n_corrected, _lang_audit_path)
    else:
        logger.info("Fichier d'audit langdetect trouve mais colonne 'langue_correcte' vide -- rien a appliquer.")
else:
    logger.info("Aucun audit langdetect anterieur trouve -- langues 100%% issues de la detection automatique.")


In [ ]:
crosstab = pd.crosstab(df[CFG.target_col], df["langue"], normalize="columns") * 100
print("Repartition des categories selon la langue (%) :")
display(crosstab.round(1))

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.heatmap(crosstab, annot=True, fmt=".1f", cmap="Blues", ax=ax, cbar_kws={"label": "%"})
ax.set_title("Repartition categorie x langue (%)")
plt.tight_layout()
plt.savefig(CFG.reports_dir / "figures" / "categorie_x_langue.png", bbox_inches="tight")
plt.show()


### 4.2 Vérification des risques de fuite (montant / tracking)

In [ ]:
AMOUNT_PATTERN = re.compile(r"\d+[\.,]?\d*\s?(?:€|eur|euros?)", flags=re.IGNORECASE)
TRACKING_PATTERN_EDA = re.compile(r"\bTRK-\w+\b", flags=re.IGNORECASE)

df["montant_detecte_texte"] = df[CFG.text_col].str.contains(AMOUNT_PATTERN, regex=True)
presence_montant = df.groupby(CFG.target_col)["montant_detecte_texte"].apply(lambda x: x.mean() * 100)
display(pd.DataFrame({"% montant mentionne dans le texte": presence_montant}).round(1))

tracking_in_text = df[CFG.text_col].str.contains(TRACKING_PATTERN_EDA, regex=True)
print(f"% de textes contenant un numero de tracking (motif TRK-xxxx) : {tracking_in_text.mean()*100:.1f}%")
print("-> masquage systematique de ce pattern en pretraitement (aucune valeur semantique retenue).")


## 5. Nettoyage et prétraitement

In [ ]:
TRACKING_PATTERN = re.compile(r"\bTRK-\w+\b", flags=re.IGNORECASE)
MULTI_SPACE_PATTERN = re.compile(r"\s+")
CENSORSHIP_PATTERN = re.compile(r"\*{2,}")

def _base_clean(text: str) -> str:
    """Nettoyage structurel commun aux deux pipelines (pas de perte de signal semantique)."""
    text = unicodedata.normalize("NFKC", str(text))
    text = TRACKING_PATTERN.sub("__TRACKING__", text)
    text = CENSORSHIP_PATTERN.sub(" __CENSURE__ ", text)
    text = MULTI_SPACE_PATTERN.sub(" ", text).strip()
    return text

def clean_text_transformer(text: str) -> str:
    """Nettoyage minimal pour DistilBERT : casse/ponctuation preservees (signal utile au tokenizer BPE)."""
    return _base_clean(text)

def clean_text_classic(text: str) -> str:
    """Nettoyage appuye pour TF-IDF : lowercase, ponctuation isolee retiree."""
    text = _base_clean(text)
    text = text.lower()
    text = re.sub(r"[^\w\s_]", " ", text, flags=re.UNICODE)
    text = MULTI_SPACE_PATTERN.sub(" ", text).strip()
    return text

df["texte_transformer"] = df[CFG.text_col].apply(clean_text_transformer)
df["texte_classic"] = df[CFG.text_col].apply(clean_text_classic)
df[[CFG.text_col, "texte_classic", "texte_transformer"]].head(3)


### 5.1 Vérification post-nettoyage

Un texte réduit à une chaîne vide après nettoyage (ex. un message ne contenant qu'un numéro de tracking ou de la ponctuation) ferait planter `TfidfVectorizer` (vocabulaire vide) et n'apporte de toute façon aucun signal exploitable pour DistilBERT. On les détecte et on les retire ici, avant la déduplication et le split.

In [ ]:
# --- Verification post-nettoyage : aucun texte ne doit devenir vide ---
empty_after_classic = df["texte_classic"].str.strip() == ""
empty_after_transformer = df["texte_transformer"].str.strip() == ""
empty_after_cleaning = empty_after_classic | empty_after_transformer
n_empty_after_cleaning = int(empty_after_cleaning.sum())

if n_empty_after_cleaning:
    logger.warning(
        "%d texte(s) devenu(s) vide(s) apres nettoyage (classic: %d, transformer: %d) -- "
        "probablement des messages ne contenant que ponctuation/tracking/censure.",
        n_empty_after_cleaning, int(empty_after_classic.sum()), int(empty_after_transformer.sum()),
    )
    empty_clean_path = CFG.reports_dir / "textes_vides_apres_nettoyage.csv"
    df.loc[empty_after_cleaning, [CFG.text_col, "texte_classic", "texte_transformer"]].to_csv(
        empty_clean_path, index=False
    )
    logger.info("Lignes concernees exportees dans %s.", empty_clean_path)
    df = df.loc[~empty_after_cleaning].reset_index(drop=True)
    logger.info("%d lignes supprimees, %d lignes restantes.", n_empty_after_cleaning, len(df))
else:
    logger.info("Verification post-nettoyage OK : aucun texte vide apres nettoyage.")


## 6. Détection et suppression des quasi-doublons

In [ ]:
dedup_vectorizer = TfidfVectorizer(max_features=5000)
X_dedup = dedup_vectorizer.fit_transform(df["texte_classic"])

def find_near_duplicates(X, threshold: float, block_size: int = 500):
    """Similarite cosinus calculee par blocs pour eviter une matrice n x n trop volumineuse en memoire."""
    n = X.shape[0]
    pairs = []
    for start in range(0, n, block_size):
        end = min(start + block_size, n)
        sim_block = cosine_similarity(X[start:end], X)
        for local_i, global_i in enumerate(range(start, end)):
            row = sim_block[local_i]
            row[global_i] = 0
            candidates = np.where(row >= threshold)[0]
            for j in candidates:
                if j > global_i:
                    pairs.append((global_i, j, row[j]))
    return pairs

near_dup_pairs = find_near_duplicates(X_dedup, CFG.near_dup_similarity_threshold)
print(f"Paires quasi-dupliquees detectees (seuil={CFG.near_dup_similarity_threshold}) : {len(near_dup_pairs)}")

# --- Correction : ne pas supprimer aveuglement sur la seule similarite textuelle ---
# Deux textes tres proches mais de categories DIFFERENTES sont soit un vrai cas limite
# entre classes qui se chevauchent (signal utile a garder), soit un faux positif du
# seuil de similarite -- dans les deux cas, une suppression automatique est risquee.
same_cat_pairs = [(i, j, s) for i, j, s in near_dup_pairs if df.loc[i, CFG.target_col] == df.loc[j, CFG.target_col]]
cross_cat_pairs = [(i, j, s) for i, j, s in near_dup_pairs if df.loc[i, CFG.target_col] != df.loc[j, CFG.target_col]]

print(f"  -> meme categorie (suppression automatique)      : {len(same_cat_pairs)}")
print(f"  -> categories DIFFERENTES (jamais supprimees auto) : {len(cross_cat_pairs)}")

if cross_cat_pairs:
    cross_cat_df = pd.DataFrame(cross_cat_pairs, columns=["idx_a", "idx_b", "similarite"])
    cross_cat_df["categorie_a"] = df.loc[cross_cat_df["idx_a"], CFG.target_col].values
    cross_cat_df["categorie_b"] = df.loc[cross_cat_df["idx_b"], CFG.target_col].values
    cross_cat_df["texte_a"] = df.loc[cross_cat_df["idx_a"], CFG.text_col].values
    cross_cat_df["texte_b"] = df.loc[cross_cat_df["idx_b"], CFG.text_col].values
    cross_cat_path = CFG.reports_dir / "near_duplicates_categories_differentes.csv"
    cross_cat_df.to_csv(cross_cat_path, index=False)
    logger.warning(
        "%d paires quasi-dupliquees relient DEUX categories differentes -- non supprimees "
        "automatiquement. A auditer manuellement : %s", len(cross_cat_pairs), cross_cat_path,
    )

indices_to_drop = sorted({j for _, j, _ in (same_cat_pairs if CFG.near_dup_require_same_category else near_dup_pairs)})
df_clean = df.drop(index=indices_to_drop).reset_index(drop=True)
print(f"Dataset : {len(df)} -> {len(df_clean)} lignes apres deduplication ({len(indices_to_drop)} lignes retirees)")
df_clean.to_csv(CFG.interim_dir / "reclamations_dedup.csv", index=False)


## 7. Split stratifié conjoint catégorie x langue

In [ ]:
df_clean["strate"] = df_clean[CFG.target_col] + "__" + df_clean["langue"]

strata_counts = df_clean["strate"].value_counts()
small_strata = strata_counts[strata_counts < CFG.min_stratum_size_warning]
if len(small_strata):
    logger.warning("Strates avec moins de %d exemples : %s", CFG.min_stratum_size_warning, dict(small_strata))

train_val_df, test_df = train_test_split(
    df_clean, test_size=CFG.test_size, stratify=df_clean["strate"], random_state=CFG.seed,
)
val_relative_size = CFG.val_size / (1 - CFG.test_size)
train_df, val_df = train_test_split(
    train_val_df, test_size=val_relative_size, stratify=train_val_df["strate"], random_state=CFG.seed,
)
print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")

def strate_proportions(d):
    return (d["strate"].value_counts(normalize=True) * 100).round(2)

control_table = pd.DataFrame({
    "train": strate_proportions(train_df), "val": strate_proportions(val_df), "test": strate_proportions(test_df),
}).fillna(0.0)
display(control_table)

# --- Garde-fou anti-fuite (bonne pratique) ---
# 1) Aucun index ne doit apparaitre dans deux splits (filet de securite si le code de
#    split est modifie plus tard).
# 2) Alerte si un texte EXACTEMENT identique se retrouve dans deux splits differents :
#    la deduplication (section 6) ne retire que les quasi-doublons au-dessus du seuil
#    de similarite, un texte strictement identique pourrait en theorie y survivre.
assert set(train_df.index).isdisjoint(val_df.index), "Fuite d'index train/val detectee !"
assert set(train_df.index).isdisjoint(test_df.index), "Fuite d'index train/test detectee !"
assert set(val_df.index).isdisjoint(test_df.index), "Fuite d'index val/test detectee !"

for name_a, d_a, name_b, d_b in [
    ("train", train_df, "val", val_df),
    ("train", train_df, "test", test_df),
    ("val", val_df, "test", test_df),
]:
    overlap_texts = set(d_a[CFG.text_col]) & set(d_b[CFG.text_col])
    if overlap_texts:
        logger.warning(
            "%d textes EXACTEMENT identiques presents a la fois dans %s et %s -- "
            "fuite potentielle malgre la deduplication (section 6).",
            len(overlap_texts), name_a, name_b,
        )
    else:
        logger.info("Aucun texte identique entre %s et %s.", name_a, name_b)

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    d.to_csv(CFG.processed_dir / f"{name}.csv", index=False)
logger.info("Splits sauvegardes dans %s", CFG.processed_dir)


## 8. Poids de classe (class weighting)

In [ ]:
classes = sorted(train_df[CFG.target_col].unique())
class_weights = compute_class_weight(class_weight="balanced", classes=np.array(classes), y=train_df[CFG.target_col])
class_weight_dict = dict(zip(classes, class_weights))

for c, w in sorted(class_weight_dict.items(), key=lambda x: -x[1]):
    print(f"  {c:30s} -> {w:.3f}")


## 9. Fonctions d'évaluation partagées (LR et DistilBERT)

In [ ]:
def evaluate_predictions(y_true, y_pred, langues, model_name: str) -> dict:
    """Metriques globales + ventilees par langue, meme format pour tous les modeles."""
    results = {"model": model_name}
    results["global"] = {
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }
    results["par_langue"] = {}
    langues = pd.Series(langues).reset_index(drop=True)
    y_true_s = pd.Series(y_true).reset_index(drop=True)
    y_pred_s = pd.Series(y_pred).reset_index(drop=True)
    for lang in ["fr", "en"]:
        mask = langues == lang
        if mask.sum() == 0:
            continue
        results["par_langue"][lang] = {
            "n": int(mask.sum()),
            "f1_macro": f1_score(y_true_s[mask], y_pred_s[mask], average="macro", zero_division=0),
        }
    return results

def print_evaluation(results: dict) -> None:
    print(f"=== {results['model']} ===")
    print(f"F1-macro global     : {results['global']['f1_macro']:.4f}")
    print(f"F1-weighted global  : {results['global']['f1_weighted']:.4f}")
    print(f"Balanced accuracy   : {results['global']['balanced_accuracy']:.4f}")
    for lang, m in results["par_langue"].items():
        print(f"F1-macro ({lang}, n={m['n']:4d}) : {m['f1_macro']:.4f}")

def plot_confusion(y_true, y_pred, labels, title: str, save_name: str) -> None:
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(7, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(CFG.reports_dir / "figures" / save_name, bbox_inches="tight")
    plt.show()


## 10. Baseline : TF-IDF + Logistic Regression

**Recherche d'hyperparamétres conjointe** (vectorizer + classifieur) : plutôt que de figer `TfidfVectorizer` puis ne chercher que `C`/`penalty` côté `LogisticRegression`, on encapsule les deux dans un `Pipeline` et on tire `RandomizedSearchCV` sur un espace conjoint (`ngram_range`, `max_features`, `min_df`, `max_df` côté vectorizer ; `C`, `class_weight` côté classifieur). Cela évite de figer arbitrairement des hyperparamétres de vectorisation qui interagissent fortement avec la régularisation du classifieur.

In [ ]:
# --- Pipeline de base ---
tfidf_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        sublinear_tf=True,   # attenue l'effet des termes tres frequents (log(1+tf))
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        random_state=CFG.seed,
    )),
])

# --- Espace de recherche conjoint vectorizer + classifieur ---
param_distributions = {
    "tfidf__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "tfidf__max_features": [3000, 5000, 10000, 30000, None],
    "tfidf__min_df": [1, 2, 3, 5],
    "tfidf__max_df": [0.85, 0.90, 0.95, 1.0],
    "clf__C": loguniform(1e-2, 1e2),
    "clf__class_weight": [None, "balanced"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG.seed)

search = RandomizedSearchCV(
    estimator=tfidf_pipeline,
    param_distributions=param_distributions,
    n_iter=CFG.baseline_search_n_iter,
    cv=cv,
    scoring="f1_macro",
    random_state=CFG.seed,
    n_jobs=-1,
    verbose=1,
)
search.fit(train_df["texte_classic"], train_df[CFG.target_col])

print(f"Meilleurs hyperparametres : {search.best_params_}")
print(f"F1-macro CV (5 plis, {CFG.baseline_search_n_iter} tirages) : {search.best_score_:.4f}")

best_baseline_model = search.best_estimator_


In [ ]:
y_train, y_val, y_test = train_df[CFG.target_col], val_df[CFG.target_col], test_df[CFG.target_col]

y_val_pred_lr = best_baseline_model.predict(val_df["texte_classic"])
print_evaluation(evaluate_predictions(y_val, y_val_pred_lr, val_df["langue"], "Logistic Regression (val)"))


In [ ]:
print(classification_report(y_val, y_val_pred_lr, zero_division=0))


### 10.1 Interprétabilité : termes les plus discriminants par classe

Avantage clé de la baseline sur un modèle de type transformer : les coefficients appris sont directement inspectables pour comprendre *pourquoi* le modèle prédit une classe donnée -- utile pour l'explicabilité, l'audit, et la confiance métier.

In [ ]:
tfidf_vectorizer = best_baseline_model.named_steps["tfidf"]
logreg = best_baseline_model.named_steps["clf"]
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())

TOP_N = 8
print("Top mots/n-grammes les plus discriminants par classe :\n")
for class_idx, class_name in enumerate(logreg.classes_):
    coefs = logreg.coef_[class_idx]
    top_idx = np.argsort(coefs)[-TOP_N:][::-1]
    top_terms = feature_names[top_idx]
    print(f"{class_name:30s}: {', '.join(top_terms)}")


In [ ]:
import joblib

joblib.dump(best_baseline_model, CFG.models_dir / "baseline_tfidf_logreg_pipeline.joblib")
logger.info("Pipeline baseline (vectorizer + LR) sauvegarde dans %s", CFG.models_dir)


## 11. Fine-tuning DistilBERT multilingue

**Modèle** : `distilbert-base-multilingual-cased` (gère nativement FR+EN dans le même espace d'embeddings, y compris le code-switching).

**Méthodologie** :

- **Validation croisée stratifiée à 5 plis sur le train set** (au lieu d'un seul split train/val fixe) : donne une estimation de performance moins sensible au bruit d'un petit échantillon de validation, et un écart-type entre plis qui quantifie directement l'instabilité du modèle sur ce volume de données.
- **Learning rates discriminants par groupe de couches** : les couches basses (embeddings, connaissances linguistiques génériques) sont modifiées très peu ; la tête de classification (initialisée aléatoirement) apprend vite. Limite le risque d'écraser le pré-entraînement avec seulement ~2000 exemples.
- **Label smoothing + gradient clipping** : régularisations supplémentaires pertinentes sur petit volume.
- **Sélection du meilleur modèle sur `eval_loss`** (pas `f1_macro`, qui peut être bruité et trompeur sur un petit val set).
- **Test set toujours tenu à l'écart** : la CV et le refit final n'utilisent que train+val ; `test_df` ne sert qu'à la comparaison finale LR vs BERT (section 14).

Sur Colab, activez un runtime GPU (`Exécution > Modifier le type d'exécution > T4 GPU`) pour un temps d'entraînement raisonnable.

In [ ]:
import torch
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU utilise :", torch.cuda.get_device_name(0))
else:
    print("-> Aucun GPU detecte : l'entrainement tournera sur CPU (beaucoup plus lent).")


In [ ]:
# --- Re-application du seeding maintenant que torch est disponible (cf. cellule Config) ---
seed_everything(CFG.seed, CFG.deterministic)

MODEL_NAME = "distilbert-base-multilingual-cased"

# --- Epinglage de la revision (bonne pratique reproductibilite / supply-chain) ---
# "main" suit la derniere version du Hub : le modele peut changer silencieusement entre deux
# executions (poids differents => resultats non reproductibles). On resout "main" vers un
# hash de commit concret ICI, une seule fois, puis on l'utilise pour TOUS les chargements
# (tokenizer + modele, y compris chaque pli de la CV). Le hash resolu est aussi sauvegarde
# dans les rapports pour tracabilite -- a reporter en dur dans MODEL_REVISION pour un usage
# en production totalement reproductible (au lieu de re-resoudre "main" a chaque run).
try:
    from huggingface_hub import HfApi
    MODEL_REVISION = HfApi().model_info(MODEL_NAME, revision="main").sha
    logger.info("Revision Hugging Face epinglee dynamiquement : %s -> %s", MODEL_NAME, MODEL_REVISION)
except Exception as exc:
    MODEL_REVISION = "main"
    logger.warning(
        "Impossible de resoudre un hash de commit precis (%s) -- repli sur 'main'. "
        "A corriger avant tout usage en production.", type(exc).__name__,
    )

with open(CFG.reports_dir / "model_revision_pinned.json", "w") as f:
    json.dump({"model_name": MODEL_NAME, "revision": MODEL_REVISION}, f, indent=2)

MAX_LENGTH = 256
BATCH_SIZE = 16
NUM_EPOCHS = 8            # cf. diagnostic precedent : la val loss se stabilise vers l'epoque 7-8 a lr=1e-5

# --- Learning rates discriminants (discriminative fine-tuning) ---
# Couches basses (embeddings + premieres couches) : representations linguistiques generiques
# deja bien apprises au pre-entrainement -> les modifier tres peu evite le catastrophic
# forgetting avec seulement ~2000 exemples de train. Tete de classification : initialisee
# aleatoirement, doit apprendre vite. LR croissant du bas vers le haut du reseau.
LR_HEAD = 5e-5
LR_TOP_LAYERS = 2e-5
LR_BASE_LAYERS = 5e-6
NUM_TOP_LAYERS = 2

# --- Regularisation ---
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2

N_SPLITS_CV = 5           # nombre de plis pour le diagnostic de stabilite

os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")

def load_with_retries(loader_fn, *args, max_retries: int = 4, initial_backoff: float = 5.0, **kwargs):
    """Retry + backoff exponentiel pour les appels from_pretrained (pannes reseau transitoires)."""
    last_exc = None
    for attempt in range(1, max_retries + 1):
        try:
            return loader_fn(*args, **kwargs)
        except Exception as exc:
            last_exc = exc
            if attempt == max_retries:
                break
            wait = initial_backoff * (2 ** (attempt - 1))
            logger.warning("Echec chargement (tentative %d/%d) : %s -- nouvel essai dans %.0fs",
                            attempt, max_retries, type(exc).__name__, wait)
            time.sleep(wait)
    raise RuntimeError(
        f"Impossible de charger '{MODEL_NAME}' depuis le Hugging Face Hub apres {max_retries} tentatives."
    ) from last_exc


In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from datasets import Dataset
import torch.nn as nn
import math

label2id = {c: i for i, c in enumerate(classes)}
id2label = {i: c for c, i in label2id.items()}

tokenizer = load_with_retries(AutoTokenizer.from_pretrained, MODEL_NAME, revision=MODEL_REVISION)

def to_hf_dataset(d: pd.DataFrame) -> Dataset:
    """Convertit un DataFrame en Dataset HF tokenise. Pas de padding ici : gere
    dynamiquement par le DataCollator au moment de construire chaque batch (plus efficace
    qu'un padding fixe a MAX_LENGTH systematiquement)."""
    ds = Dataset.from_dict({
        "text": d["texte_transformer"].tolist(),
        "label": [label2id[c] for c in d[CFG.target_col]],
    })
    ds = ds.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH), batched=True)
    ds = ds.rename_column("label", "labels")
    ds = ds.remove_columns(["text"])
    ds.set_format("torch")
    return ds

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

ds_train = to_hf_dataset(train_df)
ds_val = to_hf_dataset(val_df)
ds_test = to_hf_dataset(test_df)

class_weights_tensor_full = torch.tensor(
    [class_weight_dict[id2label[i]] for i in range(len(classes))], dtype=torch.float,
)


In [ ]:
def compute_metrics(eval_pred) -> dict:
    """F1-macro rapporte pour le suivi humain, mais PAS utilise comme critere de selection
    du meilleur modele (peut etre bruite sur un petit val set et mener a retenir un modele
    sur-ajuste). Le critere de selection reste eval_loss (cf. TrainingArguments)."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(labels, preds) if len(set(labels)) > 1 else 0.0,
    }


def build_grouped_optimizer(model: nn.Module, class_weights: torch.Tensor) -> AdamW:
    """AdamW a 6 groupes de parametres : {tete, N dernieres couches, reste+embeddings} x
    {avec weight_decay, sans}. Pattern de nommage generique "layer.N." (regex dans le code)
    pour couvrir a la fois les architectures type BERT/RoBERTa ('encoder.layer.N') et
    DistilBERT ('transformer.layer.N'). La tete de classification de DistilBERT comprend 2
    modules ('pre_classifier' ET 'classifier', contrairement a RoBERTa/CamemBERT qui n'ont
    que 'classifier') -- les deux sont donc inclus dans le bucket 'head'."""
    num_layers = getattr(model.config, "num_hidden_layers", None) or getattr(model.config, "n_layers")
    top_layer_start = num_layers - NUM_TOP_LAYERS

    no_decay_keywords = ("bias", "LayerNorm.weight", "layer_norm.weight")
    head_prefixes = ("classifier", "pre_classifier")
    layer_pattern = re.compile(r"\.layer\.(\d+)\.")

    buckets = {
        "head_decay": [], "head_no_decay": [],
        "top_decay": [], "top_no_decay": [],
        "base_decay": [], "base_no_decay": [],
    }

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        no_decay = any(kw in name for kw in no_decay_keywords)
        if name.startswith(head_prefixes):
            bucket = "head_no_decay" if no_decay else "head_decay"
        else:
            match = layer_pattern.search(name)
            if match and int(match.group(1)) >= top_layer_start:
                bucket = "top_no_decay" if no_decay else "top_decay"
            else:
                bucket = "base_no_decay" if no_decay else "base_decay"
        buckets[bucket].append(param)

    lr_map = {"head": LR_HEAD, "top": LR_TOP_LAYERS, "base": LR_BASE_LAYERS}
    optimizer_grouped_parameters = []
    for bucket_name, params in buckets.items():
        if not params:
            continue
        group_key = bucket_name.split("_")[0]
        wd = 0.0 if "no_decay" in bucket_name else WEIGHT_DECAY
        optimizer_grouped_parameters.append({"params": params, "lr": lr_map[group_key], "weight_decay": wd})

    return AdamW(optimizer_grouped_parameters)


def build_model() -> AutoModelForSequenceClassification:
    return load_with_retries(
        AutoModelForSequenceClassification.from_pretrained,
        MODEL_NAME, num_labels=len(classes), id2label=id2label, label2id=label2id, revision=MODEL_REVISION,
    ).to("cuda" if torch.cuda.is_available() else "cpu")


def build_scheduler(optimizer, num_training_steps: int):
    num_warmup_steps = int(num_training_steps * WARMUP_RATIO)
    return get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_training_steps)


def make_weighted_trainer_class(class_weights: torch.Tensor):
    """Fabrique une classe Trainer avec loss ponderee par classe + label smoothing combines
    dans un seul CrossEntropyLoss (les deux sont des arguments natifs de nn.CrossEntropyLoss
    depuis torch>=1.10, pas besoin de les combiner manuellement)."""
    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            logits = outputs.logits
            loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device), label_smoothing=LABEL_SMOOTHING)
            loss = loss_fct(logits, labels)
            return (loss, outputs) if return_outputs else loss
    return WeightedTrainer


### 11.1 Diagnostic de stabilité : validation croisée à 5 plis (sur le train set uniquement)

Objectif : estimer la performance et sa variance sur ce volume de données, sans consommer le val/test set. Chaque pli reconstruit un modèle **neuf** (poids pré-entraînés + tête réinitialisée) pour éviter toute fuite d'un pli à l'autre.

In [ ]:
from sklearn.model_selection import StratifiedKFold as StratifiedKFoldCV

fold_results = []

if not CFG.run_bert_cv_diagnostic:
    logger.warning(
        "CFG.run_bert_cv_diagnostic=False -> diagnostic de stabilite (5 entrainements complets "
        "supplementaires) saute. A activer si le quota GPU le permet : c'est la meilleure facon "
        "de detecter une forte sensibilite au split (cf. section 11.1)."
    )
else:
    skf = StratifiedKFoldCV(n_splits=N_SPLITS_CV, shuffle=True, random_state=CFG.seed)

    X_cv = train_df["texte_transformer"].values
    y_cv_strate = train_df["strate"].values  # stratification categorie x langue, coherente avec le split initial

    for fold_idx, (fold_train_idx, fold_val_idx) in enumerate(skf.split(X_cv, y_cv_strate), start=1):
        print(f"\n{'='*60}\nFOLD {fold_idx}/{N_SPLITS_CV}\n{'='*60}")

        fold_train_df = train_df.iloc[fold_train_idx].reset_index(drop=True)
        fold_val_df = train_df.iloc[fold_val_idx].reset_index(drop=True)

        fold_train_ds = to_hf_dataset(fold_train_df)
        fold_val_ds = to_hf_dataset(fold_val_df)

        fold_class_weights = torch.tensor(
            [class_weight_dict.get(id2label[i], 1.0) for i in range(len(classes))], dtype=torch.float,
        )

        fold_model = build_model()
        fold_optimizer = build_grouped_optimizer(fold_model, fold_class_weights)
        fold_steps_per_epoch = math.ceil(len(fold_train_ds) / BATCH_SIZE)
        fold_scheduler = build_scheduler(fold_optimizer, fold_steps_per_epoch * NUM_EPOCHS)

        fold_args = TrainingArguments(
            output_dir=str(CFG.models_dir / f"distilbert_cv_fold{fold_idx}"),
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            max_grad_norm=MAX_GRAD_NORM,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            save_total_limit=1,
            logging_strategy="epoch",
            fp16=torch.cuda.is_available(),
            seed=CFG.seed,
            report_to="none",
            disable_tqdm=False,
        )

        WeightedTrainerCV = make_weighted_trainer_class(fold_class_weights)
        fold_trainer = WeightedTrainerCV(
            model=fold_model, args=fold_args, train_dataset=fold_train_ds, eval_dataset=fold_val_ds,
            data_collator=data_collator, compute_metrics=compute_metrics,
            optimizers=(fold_optimizer, fold_scheduler),
            callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
        )
        fold_trainer.train()
        fold_metrics = fold_trainer.evaluate()

        fold_val_langues = fold_val_df["langue"].reset_index(drop=True)
        fold_logits = fold_trainer.predict(fold_val_ds).predictions
        fold_preds = [id2label[i] for i in np.argmax(fold_logits, axis=1)]
        fold_eval = evaluate_predictions(fold_val_df[CFG.target_col], fold_preds, fold_val_langues, f"fold_{fold_idx}")

        print(f"Fold {fold_idx} -> eval_loss={fold_metrics['eval_loss']:.4f} | f1_macro={fold_metrics['eval_f1_macro']:.4f}")

        fold_results.append({
            "fold": fold_idx,
            "eval_loss": fold_metrics["eval_loss"],
            "f1_macro": fold_eval["global"]["f1_macro"],
            "f1_macro_fr": fold_eval["par_langue"].get("fr", {}).get("f1_macro", np.nan),
            "f1_macro_en": fold_eval["par_langue"].get("en", {}).get("f1_macro", np.nan),
        })

        del fold_model, fold_trainer, fold_optimizer, fold_scheduler
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
if not fold_results:
    print("Diagnostic de stabilite saute (CFG.run_bert_cv_diagnostic=False) -- pas de resultats a afficher.")
else:
    cv_results_df = pd.DataFrame(fold_results).set_index("fold")
    print("Resultats par pli :")
    display(cv_results_df.round(4))

    cv_summary = cv_results_df.agg(["mean", "std"]).round(4)
    print("\nResume (moyenne / ecart-type sur les 5 plis) :")
    display(cv_summary)

    print(
        "\n--- Diagnostic de stabilite ---\n"
        f"F1-macro global : {cv_summary.loc['mean', 'f1_macro']:.4f} +/- {cv_summary.loc['std', 'f1_macro']:.4f}\n"
        f"F1-macro FR     : {cv_summary.loc['mean', 'f1_macro_fr']:.4f} +/- {cv_summary.loc['std', 'f1_macro_fr']:.4f}\n"
        f"F1-macro EN     : {cv_summary.loc['mean', 'f1_macro_en']:.4f} +/- {cv_summary.loc['std', 'f1_macro_en']:.4f}\n"
        "Un ecart-type eleve (> 0.03-0.05) signale une forte sensibilite au split -- a rapporter "
        "tel quel, pas a masquer."
    )


### 11.2 Entraînement final (train set complet) + sauvegarde

Une fois la stabilité vérifiée par la CV, on entraîne le modèle de production sur `train_df` complet, avec `val_df` comme critère d'arrêt (`eval_loss`, cohérent avec la CV) -- `test_df` reste réservé à la comparaison finale (section 14).

In [ ]:
final_model = build_model()
final_optimizer = build_grouped_optimizer(final_model, class_weights_tensor_full)
final_steps_per_epoch = math.ceil(len(ds_train) / BATCH_SIZE)
final_scheduler = build_scheduler(final_optimizer, final_steps_per_epoch * NUM_EPOCHS)

final_training_args = TrainingArguments(
    output_dir=str(CFG.models_dir / "distilbert_checkpoints"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    max_grad_norm=MAX_GRAD_NORM,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",   # pas f1_macro (cf. diagnostic precedent)
    greater_is_better=False,
    save_total_limit=1,
    logging_strategy="epoch",
    fp16=torch.cuda.is_available(),
    seed=CFG.seed,
    report_to="none",
)

WeightedTrainer = make_weighted_trainer_class(class_weights_tensor_full)
trainer = WeightedTrainer(
    model=final_model,
    args=final_training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(final_optimizer, final_scheduler),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

trainer.train()
trainer.save_model(str(CFG.models_dir / "distilbert_final"))
tokenizer.save_pretrained(str(CFG.models_dir / "distilbert_final"))


In [ ]:
val_preds_logits = trainer.predict(ds_val).predictions
y_val_pred_bert = [id2label[i] for i in np.argmax(val_preds_logits, axis=1)]
print_evaluation(evaluate_predictions(y_val, y_val_pred_bert, val_df["langue"], "DistilBERT multilingue (val)"))
print("\n-> A comparer a l'estimation CV ci-dessus : un ecart important suggererait encore une instabilite liee au split.")


## 12. Évaluation comparative finale (test set)

In [ ]:
y_test_pred_lr = best_baseline_model.predict(test_df["texte_classic"])
results_lr_test = evaluate_predictions(y_test, y_test_pred_lr, test_df["langue"], "Logistic Regression (test)")
print_evaluation(results_lr_test)

test_preds_logits = trainer.predict(ds_test).predictions
y_test_pred_bert = [id2label[i] for i in np.argmax(test_preds_logits, axis=1)]
results_bert_test = evaluate_predictions(y_test, y_test_pred_bert, test_df["langue"], "DistilBERT multilingue (test)")
print()
print_evaluation(results_bert_test)

final_results = [results_lr_test, results_bert_test]


In [ ]:
comparison_rows = []
for r in final_results:
    row = {
        "Modele": r["model"],
        "F1-macro global": r["global"]["f1_macro"],
        "F1-weighted global": r["global"]["f1_weighted"],
        "Balanced accuracy": r["global"]["balanced_accuracy"],
    }
    for lang, m in r["par_langue"].items():
        row[f"F1-macro ({lang})"] = m["f1_macro"]
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).round(4)
display(comparison_df)
comparison_df.to_csv(CFG.reports_dir / "comparaison_finale_modeles.csv", index=False)


### 12.1 Fiche modèle (model card) — traçabilité

On regroupe dans un seul fichier JSON tout ce qu'il faut pour savoir, plus tard, exactement quel run a produit quel modèle : métriques finales, configuration complète, date d'entraînement, hash du fichier de données source et versions des librairies. Utile pour comparer plusieurs exécutions ou auditer un modèle en production.

In [ ]:
# --- Fiche modele (model card) : un seul fichier de tracabilite pour ce run ---
import hashlib
import platform
from datetime import datetime, timezone


def _file_hash(path: Path, algo: str = "sha256") -> str:
    """Hash du fichier CSV source, pour tracer precisement quelles donnees ont produit ce modele."""
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


_baseline_params = best_baseline_model.get_params()
_baseline_retained = {
    k: str(v) for k, v in _baseline_params.items() if k.startswith(("tfidf__", "clf__"))
}

model_card = {
    "date_entrainement": datetime.now(timezone.utc).isoformat(),
    "seed": CFG.seed,
    "donnees": {
        "fichier_source": str(CFG.raw_path),
        "hash_sha256": _file_hash(CFG.raw_path) if CFG.raw_path.exists() else None,
        "n_lignes_brutes": n_before,
        "n_lignes_apres_nettoyage": len(df),
        "n_train": len(train_df),
        "n_val": len(val_df),
        "n_test": len(test_df),
    },
    "configuration": {k: (str(v) if isinstance(v, Path) else v) for k, v in vars(CFG).items()},
    "environnement": {
        "python": platform.python_version(),
        **versions_used,
    },
    "modeles": {
        "baseline": {
            "type": "TfidfVectorizer + LogisticRegression",
            "chemin": str(CFG.models_dir / "baseline_tfidf_logreg_pipeline.joblib"),
            "hyperparametres_retenus": _baseline_retained,
        },
        "distilbert": {
            "type": MODEL_NAME,
            "chemin": str(CFG.models_dir / "distilbert_final"),
        },
    },
    "metriques_test": comparison_df.to_dict(orient="records"),
}

model_card_path = CFG.reports_dir / "model_card.json"
with open(model_card_path, "w", encoding="utf-8") as f:
    json.dump(model_card, f, indent=2, ensure_ascii=False, default=str)

logger.info("Fiche modele (model card) sauvegardee dans %s", model_card_path)
print(f"Fiche modele : {model_card_path}")
print(f"  Hash SHA256 des donnees source : {model_card['donnees']['hash_sha256']}")
print(f"  Date d'entrainement (UTC)      : {model_card['date_entrainement']}")


## 13. Tests de significativité statistique et diagnostics complémentaires

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def mcnemar_pairwise(y_true, pred_a, pred_b, name_a: str, name_b: str) -> dict:
    """McNemar : compare deux modeles sur les memes exemples de test (donnees appariees)."""
    y_true = pd.Series(y_true).reset_index(drop=True)
    pred_a = pd.Series(pred_a).reset_index(drop=True)
    pred_b = pd.Series(pred_b).reset_index(drop=True)
    correct_a, correct_b = pred_a == y_true, pred_b == y_true
    both_correct = int((correct_a & correct_b).sum())
    only_a_correct = int((correct_a & ~correct_b).sum())
    only_b_correct = int((~correct_a & correct_b).sum())
    both_wrong = int((~correct_a & ~correct_b).sum())
    table = [[both_correct, only_a_correct], [only_b_correct, both_wrong]]
    n_discordant = only_a_correct + only_b_correct
    result = mcnemar(table, exact=(n_discordant < 25), correction=True)
    verdict = "SIGNIFICATIF (p<0.05)" if result.pvalue < 0.05 else "non significatif (p>=0.05)"
    print(f"=== McNemar : {name_a} vs {name_b} ===")
    print(f"  {name_a} correct seul : {only_a_correct}")
    print(f"  {name_b} correct seul : {only_b_correct}")
    print(f"  Desaccords totaux    : {n_discordant}")
    print(f"  statistic = {result.statistic:.4f} | p-value = {result.pvalue:.4f}")
    print(f"  -> {verdict}\n")
    return {"comparaison": f"{name_a} vs {name_b}", "only_a_correct": only_a_correct,
            "only_b_correct": only_b_correct, "statistic": result.statistic,
            "p_value": result.pvalue, "significatif": bool(result.pvalue < 0.05)}

mcnemar_result = mcnemar_pairwise(y_test, y_test_pred_lr, y_test_pred_bert, "LR", "BERT")


In [ ]:
def bootstrap_f1_macro(y_true, y_pred, langues=None, n_iter: int = 1000, seed: int = 42) -> dict:
    """F1-macro observe + IC95% bootstrap, global et par langue."""
    rng = np.random.default_rng(seed)
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    n = len(y_true)

    def _f1(idx):
        return f1_score(y_true.iloc[idx], y_pred.iloc[idx], average="macro", zero_division=0)

    boot_scores = np.array([_f1(rng.integers(0, n, n)) for _ in range(n_iter)])
    global_result = {
        "observed": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "ci_low": np.percentile(boot_scores, 2.5),
        "ci_high": np.percentile(boot_scores, 97.5),
        "std": boot_scores.std(),
    }
    result = {"global": global_result, "par_langue": {}}
    if langues is not None:
        langues = pd.Series(langues).reset_index(drop=True)
        for lang in ["fr", "en"]:
            mask_idx = np.where(langues == lang)[0]
            if len(mask_idx) == 0:
                continue
            n_lang = len(mask_idx)

            def _f1_lang(_, mask_idx=mask_idx, n_lang=n_lang):
                sampled = rng.choice(mask_idx, size=n_lang, replace=True)
                return f1_score(y_true.iloc[sampled], y_pred.iloc[sampled], average="macro", zero_division=0)

            boot_lang = np.array([_f1_lang(i) for i in range(n_iter)])
            result["par_langue"][lang] = {
                "n": n_lang,
                "observed": f1_score(y_true.iloc[mask_idx], y_pred.iloc[mask_idx], average="macro", zero_division=0),
                "ci_low": np.percentile(boot_lang, 2.5),
                "ci_high": np.percentile(boot_lang, 97.5),
                "std": boot_lang.std(),
            }
    return result

def print_bootstrap_result(model_name: str, result: dict) -> None:
    g = result["global"]
    print(f"=== {model_name} (bootstrap, IC95%, n_iter=1000) ===")
    print(f"  Global  : {g['observed']:.4f}  [{g['ci_low']:.4f} ; {g['ci_high']:.4f}]  (std={g['std']:.4f})")
    for lang, m in result["par_langue"].items():
        print(f"  {lang} (n={m['n']:3d}) : {m['observed']:.4f}  [{m['ci_low']:.4f} ; {m['ci_high']:.4f}]  (std={m['std']:.4f})")
    print()

predictions = {"LR": y_test_pred_lr, "BERT": y_test_pred_bert}
bootstrap_results = {}
for name, preds in predictions.items():
    bootstrap_results[name] = bootstrap_f1_macro(y_test, preds, test_df["langue"], n_iter=1000, seed=CFG.seed)
    print_bootstrap_result(name, bootstrap_results[name])

ra, rb = bootstrap_results["LR"]["global"], bootstrap_results["BERT"]["global"]
overlap = not (ra["ci_high"] < rb["ci_low"] or rb["ci_high"] < ra["ci_low"])
print("IC se chevauchent -> ecart NON concluant" if overlap else "IC disjoints -> ecart probablement reel")


In [ ]:
def diagnose_class(y_true, predictions: dict, target_class: str) -> pd.DataFrame:
    """F1/precision/recall pour une classe isolee (un F1-macro global peut la masquer)."""
    y_true = pd.Series(y_true).reset_index(drop=True)
    rows = []
    for model_name, y_pred in predictions.items():
        y_pred = pd.Series(y_pred).reset_index(drop=True)
        rows.append({
            "modele": model_name, "classe": target_class,
            "precision": precision_score(y_true, y_pred, labels=[target_class], average="macro", zero_division=0),
            "recall": recall_score(y_true, y_pred, labels=[target_class], average="macro", zero_division=0),
            "f1": f1_score(y_true, y_pred, labels=[target_class], average="macro", zero_division=0),
            "n_exemples_reels": int((y_true == target_class).sum()),
        })
    result_df = pd.DataFrame(rows)
    print(f"=== Diagnostic classe '{target_class}' (isolee, pas macro-moyennee) ===")
    print(result_df.to_string(index=False))
    return result_df

diagnose_class(y_test, predictions, target_class=classes[0])


In [ ]:
def sample_for_langdetect_audit(df, text_col, lang_col, n_per_lang=40, seed=42):
    """Echantillon stratifie a verifier manuellement (colonne langue_correcte a remplir,
    avec un code langue tel que 'fr', 'en' ou 'other'). Sera reintegre AUTOMATIQUEMENT au
    prochain run, avant le split (cf. cellule de reintegration juste apres la detection
    de langue, section 4.1) -- une correction appliquee apres le split ne changerait pas
    la stratification deja realisee."""
    rng = np.random.default_rng(seed)
    samples = []
    for lang in df[lang_col].unique():
        subset = df[df[lang_col] == lang]
        n = min(n_per_lang, len(subset))
        idx = rng.choice(subset.index, size=n, replace=False)
        samples.append(subset.loc[idx])
    audit_df = pd.concat(samples)[[text_col, lang_col]].reset_index(drop=True)
    audit_df = audit_df.rename(columns={lang_col: "langue_detectee"})
    audit_df["langue_correcte"] = ""
    audit_df["texte_court"] = audit_df[text_col].str.slice(0, 150)
    return audit_df[["texte_court", "langue_detectee", "langue_correcte"]]

audit_path = CFG.reports_dir / "audit_langdetect_a_completer.csv"
audit_df = sample_for_langdetect_audit(df, CFG.text_col, "langue", n_per_lang=40, seed=CFG.seed)
audit_df.to_csv(audit_path, index=False)
print(f"-> Completer manuellement la colonne 'langue_correcte' dans {audit_path}.")
print("   Les corrections seront appliquees AUTOMATIQUEMENT au prochain run (section 4.1), avant le split.")


## 14. Analyse des erreurs

In [ ]:
plot_confusion(y_test, y_test_pred_lr, classes, "Matrice de confusion - Logistic Regression (test)", "confusion_lr.png")


In [ ]:
plot_confusion(y_test, y_test_pred_bert, classes, "Matrice de confusion - DistilBERT (test)", "confusion_bert.png")


In [ ]:
errors_df = test_df.copy()
errors_df["prediction_lr"] = y_test_pred_lr
errors_df = errors_df[errors_df[CFG.target_col] != errors_df["prediction_lr"]]

print(f"Nombre d'erreurs (LR, test) : {len(errors_df)} / {len(test_df)} ({len(errors_df)/len(test_df)*100:.1f}%)")
display(errors_df[[CFG.target_col, "prediction_lr", "langue", CFG.text_col]].head(10))

confusion_pairs = (
    errors_df.groupby([CFG.target_col, "prediction_lr"]).size().sort_values(ascending=False).head(10)
)
print("Confusions les plus frequentes (vraie -> predite) :")
display(confusion_pairs)


### 14.1 Erreurs les plus confiantes

Une erreur commise avec une confiance de prédiction élevée est souvent plus révélatrice d'un vrai problème (exemple ambigu, catégories qui se chevauchent, biais du modèle) qu'une erreur commise avec une confiance proche du hasard. On trie donc les erreurs de chaque modèle par confiance décroissante plutôt que de les lire dans un ordre arbitraire.

In [ ]:
def build_error_dataframe(texts, y_true, y_pred, proba=None, class_order=None) -> pd.DataFrame:
    """proba : matrice (n_exemples, n_classes) alignee sur class_order (liste des classes,
    meme ordre que les colonnes de proba). Utilisee pour recuperer la confiance de la
    prediction (probabilite maximale)."""
    out = pd.DataFrame({
        "texte": pd.Series(texts).reset_index(drop=True),
        "classe_reelle": pd.Series(y_true).reset_index(drop=True),
        "classe_predite": pd.Series(y_pred).reset_index(drop=True),
    })
    if proba is not None:
        out["confiance_predite"] = np.asarray(proba).max(axis=1)
    out["erreur"] = out["classe_reelle"] != out["classe_predite"]
    return out

# --- Confiance LR : predict_proba natif du pipeline sklearn ---
lr_proba = best_baseline_model.predict_proba(test_df["texte_classic"])
lr_class_order = best_baseline_model.named_steps["clf"].classes_
errors_df_lr = build_error_dataframe(test_df[CFG.text_col], y_test, y_test_pred_lr, lr_proba, lr_class_order)

# --- Confiance DistilBERT : softmax des logits de test deja calcules en section 12 ---
bert_logits = test_preds_logits
bert_proba = np.exp(bert_logits - bert_logits.max(axis=1, keepdims=True))
bert_proba = bert_proba / bert_proba.sum(axis=1, keepdims=True)
errors_df_bert = build_error_dataframe(test_df[CFG.text_col], y_test, y_test_pred_bert, bert_proba)

print(f"Baseline LR : {errors_df_lr['erreur'].sum()} erreurs / {len(errors_df_lr)} ({errors_df_lr['erreur'].mean():.1%})")
print(f"DistilBERT  : {errors_df_bert['erreur'].sum()} erreurs / {len(errors_df_bert)} ({errors_df_bert['erreur'].mean():.1%})")


### 14.1bis Vérification de la calibration des probabilités

La section 14.1 trie les erreurs par confiance prédite (probabilité max), en supposant implicitement que cette confiance reflète une vraie probabilité de correction ("90% de confiance" ≈ correct 90% du temps). On vérifie ici cette hypothèse via une calibration "top-label" : on regroupe les prédictions (correctes ou non, toutes classes confondues) en tranches de confiance, et on compare la confiance moyenne de chaque tranche à l'exactitude empiriquement observée dedans. Un modèle mal calibré (surconfiant) peut rendre le classement des "erreurs les plus confiantes" trompeur : une confiance de 90% peut en réalité correspondre à un taux de succès bien inférieur.

In [ ]:
def top_label_calibration(y_true, y_pred, confidence, n_bins: int = 10) -> pd.DataFrame:
    """Calibration 'top-label' : regroupe les predictions par tranche de confiance et compare
    la confiance moyenne a l'exactitude empirique dans chaque tranche. Une bonne calibration
    donne des points proches de la diagonale confiance == exactitude."""
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    confidence = pd.Series(confidence).reset_index(drop=True)
    correct = (y_true == y_pred).astype(int)

    bins = pd.cut(confidence, bins=n_bins, include_lowest=True)
    grouped = pd.DataFrame({"confiance": confidence, "correct": correct, "bin": bins}).groupby("bin", observed=True)
    summary = grouped.agg(
        confiance_moyenne=("confiance", "mean"),
        exactitude_empirique=("correct", "mean"),
        n=("correct", "size"),
    ).dropna()
    summary["ecart_abs"] = (summary["confiance_moyenne"] - summary["exactitude_empirique"]).abs()
    return summary


def expected_calibration_error(summary: pd.DataFrame) -> float:
    """ECE : moyenne des ecarts |confiance - exactitude| par tranche, ponderee par effectif."""
    return float((summary["ecart_abs"] * summary["n"]).sum() / summary["n"].sum())


calib_lr = top_label_calibration(y_test, y_test_pred_lr, errors_df_lr["confiance_predite"])
calib_bert = top_label_calibration(y_test, y_test_pred_bert, errors_df_bert["confiance_predite"])

ece_lr = expected_calibration_error(calib_lr)
ece_bert = expected_calibration_error(calib_bert)

print(f"ECE (Expected Calibration Error) -- Logistic Regression : {ece_lr:.4f}")
print(f"ECE (Expected Calibration Error) -- DistilBERT           : {ece_bert:.4f}")
print("(plus proche de 0 = mieux calibre ; au-dela de ~0.05-0.10, interpreter le tri par")
print(" 'confiance' de la section 14.1 avec prudence.)")

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([0, 1], [0, 1], "k--", label="Calibration parfaite")
ax.plot(calib_lr["confiance_moyenne"], calib_lr["exactitude_empirique"], "o-", label=f"LR (ECE={ece_lr:.3f})")
ax.plot(calib_bert["confiance_moyenne"], calib_bert["exactitude_empirique"], "s-", label=f"DistilBERT (ECE={ece_bert:.3f})")
ax.set_xlabel("Confiance moyenne predite")
ax.set_ylabel("Exactitude empirique")
ax.set_title("Diagramme de fiabilite (calibration) -- test set")
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(CFG.reports_dir / "figures" / "calibration_reliability_diagram.png", dpi=150)
plt.show()


In [ ]:
print("=== Erreurs de la baseline TF-IDF + LogReg les plus 'confiantes' ===\n")
top_lr_errors = errors_df_lr[errors_df_lr["erreur"]].sort_values("confiance_predite", ascending=False).head(10)
for _, row in top_lr_errors.iterrows():
    print(f"[reel: {row['classe_reelle']:25s} | predit: {row['classe_predite']:25s} | conf: {row['confiance_predite']:.2f}]")
    print(f"  {row['texte']}\n")


In [ ]:
print("=== Erreurs de DistilBERT les plus 'confiantes' ===\n")
top_bert_errors = errors_df_bert[errors_df_bert["erreur"]].sort_values("confiance_predite", ascending=False).head(10)
for _, row in top_bert_errors.iterrows():
    print(f"[reel: {row['classe_reelle']:25s} | predit: {row['classe_predite']:25s} | conf: {row['confiance_predite']:.2f}]")
    print(f"  {row['texte']}\n")


## 15. Téléchargement des artefacts (modèles, rapports)

In [ ]:
import shutil

shutil.make_archive("/content/reclamations_outputs", "zip", root_dir="/content", base_dir="models")
shutil.make_archive("/content/reclamations_reports", "zip", root_dir="/content", base_dir="reports")

from google.colab import files
files.download("/content/reclamations_outputs.zip")
files.download("/content/reclamations_reports.zip")


## 16. Extraction d'entites (NER)

Objectif : extraire automatiquement 4 types d'entites du texte libre des reclamations : `NUMERO_TRACKING`, `VILLE`, `MONTANT`, `DATE`, via un modele spaCy custom.

**Deux points d'attention specifiques a ce notebook avant de commencer :**
1. Le CSV brut contient deja des colonnes structurees (`montant`, `localisation`, `date`, `tracking_id`) qui donnent la valeur *correcte* de chaque entite sans effort d'annotation manuelle -> on les exploite comme supervision faible (weak supervision) pour pre-annoter le texte automatiquement.
2. Le NER doit etre entraine sur le texte **brut** (`df[CFG.text_col]`), pas sur `texte_classic` ni `texte_transformer` : ces deux colonnes masquent deja les motifs `TRK-xxxx` (section 4, `_base_clean`), ce qui detruirait l'entite `NUMERO_TRACKING` si on les reutilisait ici.

### 16.1 Schema d'annotation

In [ ]:
# --- Schema d'annotation NER ---
ENTITY_LABELS = ["NUMERO_TRACKING", "VILLE", "MONTANT", "DATE"]

NER_TRACKING_PATTERN = re.compile(r"\bTRK-\w+\b", flags=re.IGNORECASE)

# Montant : "1871 euros", "1 871,00 EUR", "1871€"...
NER_AMOUNT_PATTERN = re.compile(
    r"\d{1,3}(?:[ .]\d{3})*(?:[.,]\d{1,2})?\s?(?:\u20ac|eur\b|euros?\b)", flags=re.IGNORECASE
)

# Date : format numerique (16/11/2025, 16-11-2025) ou textuel (16 novembre 2025)
NER_DATE_NUMERIC_PATTERN = re.compile(r"\b\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}\b")
MOIS_FR = ("janvier", "f\u00e9vrier", "fevrier", "mars", "avril", "mai", "juin", "juillet",
           "ao\u00fbt", "aout", "septembre", "octobre", "novembre", "d\u00e9cembre", "decembre")
NER_DATE_TEXT_PATTERN = re.compile(
    r"\b\d{1,2}\s+(?:" + "|".join(MOIS_FR) + r")\s+\d{4}\b", flags=re.IGNORECASE
)

logger.info("Schema NER defini : %s", ENTITY_LABELS)


### 16.2 Pre-annotation automatique (weak supervision)

On combine deux strategies selon l'entite :
- **Regex generique** pour `NUMERO_TRACKING`, `MONTANT`, `DATE` (formats suffisamment reguliers).
- **Correspondance litterale avec la colonne structuree** pour `VILLE` (pas de pattern generique fiable pour reconnaitre un nom de ville dans du texte libre multilingue).

In [ ]:
def _find_span(text: str, pattern) -> list:
    """Toutes les positions (start, end) d'un pattern compile dans text."""
    return [(m.start(), m.end()) for m in pattern.finditer(text)]


def _find_literal_span(text: str, value):
    """Cherche value dans text. Essaie D'ABORD une recherche directe insensible a la casse
    (indices garantis exacts) ; ne bascule sur la recherche normalisee (accents retires)
    qu'en repli. Corrige un point faible de la version precedente qui normalisait toujours,
    au risque de decaler les indices sur des caracteres qui se decomposent en plusieurs
    codepoints. Retourne (start, end) ou None."""
    if not isinstance(value, str) or not value.strip():
        return None
    value = value.strip()

    # 1) Recherche directe insensible a la casse -- indices exacts, aucune approximation.
    idx = text.lower().find(value.lower())
    if idx != -1:
        return (idx, idx + len(value))

    # 2) Repli : normalisation NFKD+ascii (retire les accents). Approximation : suppose que
    # la normalisation ne change pas la longueur de la sous-chaine -- vrai pour l'immense
    # majorite des noms de ville latins ; cas limites a surveiller via l'audit (section 16.3).
    norm_text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode().lower()
    norm_value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode().lower()
    idx = norm_text.find(norm_value)
    if idx == -1:
        return None
    return (idx, idx + len(norm_value))


def auto_annotate_row(row) -> list:
    """Pre-annotation par supervision faible sur le texte brut. Retourne une liste de
    tuples (start, end, label) sans chevauchement (le plus long span gagne en cas de conflit)."""
    text = row[CFG.text_col]
    if not isinstance(text, str):
        return []

    spans = []
    for start, end in _find_span(text, NER_TRACKING_PATTERN):
        spans.append((start, end, "NUMERO_TRACKING"))
    for start, end in _find_span(text, NER_AMOUNT_PATTERN):
        spans.append((start, end, "MONTANT"))
    for start, end in _find_span(text, NER_DATE_NUMERIC_PATTERN):
        spans.append((start, end, "DATE"))
    for start, end in _find_span(text, NER_DATE_TEXT_PATTERN):
        spans.append((start, end, "DATE"))

    if "localisation" in row.index:
        loc_span = _find_literal_span(text, row["localisation"])
        if loc_span:
            spans.append((*loc_span, "VILLE"))

    spans = sorted(set(spans), key=lambda s: (s[0], -(s[1] - s[0])))
    filtered, last_end = [], -1
    for start, end, label in spans:
        if start >= last_end:
            filtered.append((start, end, label))
            last_end = end
    return filtered


df["ner_spans"] = df.apply(auto_annotate_row, axis=1)

coverage = pd.Series({
    lbl: df["ner_spans"].apply(lambda spans: any(l == lbl for _, _, l in spans)).mean() * 100
    for lbl in ENTITY_LABELS
})
print("Couverture de la pre-annotation automatique (% de lignes avec >= 1 entite detectee) :")
print(coverage.round(1))

# --- Bonne pratique : verification de coherence vs les colonnes structurees ---
# Le CSV brut contient deja la valeur "vraie" de chaque entite (section 16 intro). Si une
# ligne a une valeur structuree non vide mais qu'AUCUN span correspondant n'a ete trouve
# dans le texte, c'est un signal de SOUS-annotation (regex trop stricte, format non couvert,
# etc.) -- a corriger dans le schema de regex plutot que de laisser un gold set incomplet.
STRUCTURED_COL_FOR_ENTITY = {
    "NUMERO_TRACKING": "tracking_id",
    "VILLE": "localisation",
    "MONTANT": "montant",
    "DATE": "date",
}

print("\nCoherence pre-annotation vs colonnes structurees (detection de sous-annotation) :")
for label, col in STRUCTURED_COL_FOR_ENTITY.items():
    if col not in df.columns:
        logger.info("Colonne structuree '%s' absente du CSV -- verification de coherence sautee pour %s.", col, label)
        continue
    has_structured = df[col].notna() & (df[col].astype(str).str.strip() != "")
    has_span = df["ner_spans"].apply(lambda spans: any(l == label for _, _, l in spans))
    missed = has_structured & ~has_span
    n_structured = int(has_structured.sum())
    n_missed = int(missed.sum())
    if n_structured == 0:
        continue
    pct = n_missed / n_structured * 100
    print(f"  {label:18s} (col '{col}') : {n_missed}/{n_structured} lignes avec valeur structuree "
          f"mais aucun span detecte ({pct:.1f}%)")
    if n_missed:
        missed_path = CFG.reports_dir / f"ner_sous_annotation_{label.lower()}.csv"
        df.loc[missed, [CFG.text_col, col]].head(50).to_csv(missed_path, index=False)
        logger.warning("%s : %d cas de sous-annotation exportes dans %s", label, n_missed, missed_path)


### 16.3 Audit manuel de la pre-annotation

Meme logique que l'audit `langdetect` (section 13) : on ne fait pas confiance aveuglement a la pre-annotation automatique, on en verifie un echantillon.

In [ ]:
def sample_for_ner_audit(df, text_col, spans_col, n=60, seed=42):
    """Echantillon aleatoire a verifier manuellement. Format attendu pour la colonne
    'annotation_correcte' :
      - vide ou 'OK'                  -> pre-annotation jugee correcte, aucun changement
      - liste JSON de spans corriges  -> ex. [[12, 20, "MONTANT"], [0, 5, "VILLE"]]
                                          remplace INTEGRALEMENT les spans de cette ligne
    'row_index' permet de recoller la correction a la bonne ligne du DataFrame d'origine
    (cf. cellule de reintegration juste avant la conversion au format spaCy, section 16.4)."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(df.index, size=min(n, len(df)), replace=False)
    audit_df = df.loc[idx, [text_col, spans_col]].reset_index()
    audit_df = audit_df.rename(columns={"index": "row_index"})
    audit_df["annotation_correcte"] = ""
    return audit_df


ner_audit_path = CFG.reports_dir / "audit_ner_a_completer.csv"
sample_for_ner_audit(df, CFG.text_col, "ner_spans", n=60, seed=CFG.seed).to_csv(ner_audit_path, index=False)
print(f"-> Completer manuellement 'annotation_correcte' dans {ner_audit_path} avant l'entrainement.")
print("   Format : vide/'OK' = pre-annotation correcte, sinon liste JSON [[start,end,label], ...].")
print("   Les corrections seront reintegrees automatiquement au prochain run (avant la section 16.4).")


### 16.3bis Reintegration des corrections de l'audit NER (avant entrainement)

Bonne pratique : une correction manuelle qui n'est jamais rebranchee dans le pipeline ne sert a rien. On applique ici les corrections de `audit_ner_a_completer.csv` **avant** de generer les fichiers `.spacy` d'entrainement -- sinon elles n'auraient aucun effet sur le modele.


In [ ]:
# --- Reintegration des corrections d'un audit NER manuel precedent ---
_ner_audit_path = CFG.reports_dir / "audit_ner_a_completer.csv"
_n_ner_corrected = 0

if _ner_audit_path.exists():
    _ner_audit = pd.read_csv(_ner_audit_path)
    for _, _row in _ner_audit.iterrows():
        _correction = str(_row.get("annotation_correcte", "")).strip()
        if _correction.lower() in ("", "nan", "ok"):
            continue
        try:
            _new_spans = [tuple(span) for span in json.loads(_correction)]
        except (json.JSONDecodeError, TypeError):
            logger.warning("Correction NER illisible (ligne %s), ignoree : %r", _row["row_index"], _correction)
            continue
        df.at[int(_row["row_index"]), "ner_spans"] = _new_spans
        _n_ner_corrected += 1
    logger.info("%d lignes NER corrigees depuis l'audit manuel (%s)", _n_ner_corrected, _ner_audit_path)
else:
    logger.info("Aucun audit NER anterieur trouve -- spans 100%% issus de la pre-annotation automatique.")

if _n_ner_corrected == 0:
    logger.warning(
        "Le score NER (section 16.6) reste mesure contre une pre-annotation heuristique, pas une "
        "verite terrain independante -- a mentionner explicitement comme limite dans le rapport final."
    )


### 16.4 Conversion au format spaCy et split train/dev/test

On reutilise le **meme split stratifie categorie x langue** que pour la classification (`train_df` / `val_df` / `test_df`, section 6), pour rester coherent entre les deux taches.

In [ ]:
import spacy
from spacy.tokens import DocBin
from spacy.util import filter_spans

def to_spacy_docbin(sub_df: pd.DataFrame, nlp) -> DocBin:
    """Convertit un DataFrame (texte brut + ner_spans) en DocBin spaCy. alignment_mode='contract'
    retrecit un span aux frontieres de token les plus proches plutot que de planter si l'alignement
    caractere/token n'est pas exact."""
    doc_bin = DocBin()
    n_skipped = 0
    for _, row in sub_df.iterrows():
        doc = nlp.make_doc(row[CFG.text_col])
        ents = []
        for start, end, label in row["ner_spans"]:
            span = doc.char_span(start, end, label=label, alignment_mode="contract")
            if span is not None:
                ents.append(span)
            else:
                n_skipped += 1
        doc.ents = filter_spans(ents)  # supprime les chevauchements residuels
        doc_bin.add(doc)
    if n_skipped:
        logger.warning("%d spans ignores (alignement token impossible)", n_skipped)
    return doc_bin


nlp_blank = spacy.blank("fr")  # tokenizer francais ; pas de composants pre-entraines a ce stade

ner_train_df = df.loc[df.index.isin(train_df.index)]
ner_val_df = df.loc[df.index.isin(val_df.index)]
ner_test_df = df.loc[df.index.isin(test_df.index)]

ner_data_dir = CFG.processed_dir / "ner"
ner_data_dir.mkdir(parents=True, exist_ok=True)

to_spacy_docbin(ner_train_df, nlp_blank).to_disk(ner_data_dir / "train.spacy")
to_spacy_docbin(ner_val_df, nlp_blank).to_disk(ner_data_dir / "dev.spacy")
to_spacy_docbin(ner_test_df, nlp_blank).to_disk(ner_data_dir / "test.spacy")

print(f"Train : {len(ner_train_df)} | Dev : {len(ner_val_df)} | Test : {len(ner_test_df)}")


### 16.4bis Distribution des types d'entites par split

Meme logique que pour les categories de reclamation (section 7) : un type d'entite absent ou tres rare dans le train, le dev ou le test fausse son evaluation -- ex. 0 exemple `VILLE` dans `dev` rendrait le rappel `VILLE` du diagnostic (section 16.6) non interpretable.


In [ ]:
def entity_distribution(sub_df: pd.DataFrame) -> pd.Series:
    counts = {lbl: 0 for lbl in ENTITY_LABELS}
    for spans in sub_df["ner_spans"]:
        for _, _, label in spans:
            counts[label] += 1
    return pd.Series(counts)

ner_dist = pd.DataFrame({
    "train": entity_distribution(ner_train_df),
    "dev": entity_distribution(ner_val_df),
    "test": entity_distribution(ner_test_df),
})
print("Nombre d'occurrences par type d'entite et par split :")
display(ner_dist)

for lbl in ENTITY_LABELS:
    zero_splits = [name for name in ["train", "dev", "test"] if ner_dist.loc[lbl, name] == 0]
    if zero_splits:
        logger.warning(
            "Type d'entite '%s' absent de : %s -- evaluation non fiable pour ce type sur ce(s) split(s).",
            lbl, zero_splits,
        )


### 16.5 Configuration et entrainement du modele NER custom

`spacy init config` genere une config de base (preset `efficiency` = CPU-friendly, convient a un pipeline NER seul sur un volume de donnees modeste ; passer a `accuracy` si le volume et le GPU le justifient).

In [ ]:
!python -m spacy init config {CFG.models_dir}/ner_config.cfg --lang fr --pipeline ner --optimize efficiency --force


In [ ]:
!python -m spacy train {CFG.models_dir}/ner_config.cfg \
    --output {CFG.models_dir}/ner \
    --paths.train {ner_data_dir}/train.spacy \
    --paths.dev {ner_data_dir}/dev.spacy \
    --training.seed {CFG.seed} \
    --gpu-id 0


### 16.6 Evaluation sur le set de test

In [ ]:
from spacy.scorer import Scorer
from spacy.training import Example

nlp_ner = spacy.load(CFG.models_dir / "ner" / "model-best")

def evaluate_ner(nlp, sub_df: pd.DataFrame) -> dict:
    """Score precision/recall/F1 global + par type d'entite, via le Scorer natif de spaCy
    (coherent avec la sortie de `spacy evaluate`)."""
    examples = []
    for _, row in sub_df.iterrows():
        pred_doc = nlp(row[CFG.text_col])
        gold_doc = nlp.make_doc(row[CFG.text_col])
        ents = []
        for start, end, label in row["ner_spans"]:
            span = gold_doc.char_span(start, end, label=label, alignment_mode="contract")
            if span is not None:
                ents.append(span)
        gold_doc.ents = filter_spans(ents)
        examples.append(Example(pred_doc, gold_doc))
    return Scorer().score(examples)


ner_scores = evaluate_ner(nlp_ner, ner_test_df)

print(f"Precision globale : {ner_scores['ents_p']:.3f}")
print(f"Recall globale    : {ner_scores['ents_r']:.3f}")
print(f"F1 globale        : {ner_scores['ents_f']:.3f}")
print()
print("Par type d'entite (un F1 global peut masquer une entite rare qui performe mal) :")
per_type_scores = pd.DataFrame(ner_scores["ents_per_type"]).T
display(per_type_scores.round(3))

per_type_scores.to_csv(CFG.reports_dir / "ner_scores_par_entite.csv")


### 16.6bis Comparaison a une baseline regex seule

Bonne pratique : 3 des 4 types d'entites (`NUMERO_TRACKING`, `MONTANT`, `DATE`) sont deja capturables par les memes regex utilisees pour la pre-annotation (section 16.2). Sans comparaison, on ne sait pas si le modele spaCy apporte une vraie valeur ajoutee par rapport a un simple regex -- ce qui orienterait vers un pipeline plus simple et moins couteux en production. Le ML n'est vraiment necessaire que pour `VILLE`, sans pattern generique fiable.


In [ ]:
def regex_only_predict(text: str) -> list:
    """Meme logique que auto_annotate_row (section 16.2), mais SANS VILLE : aucun pattern
    generique fiable ne permet de la detecter sans le modele spaCy (ou la colonne structuree)."""
    return (
        [(s, e, "NUMERO_TRACKING") for s, e in _find_span(text, NER_TRACKING_PATTERN)]
        + [(s, e, "MONTANT") for s, e in _find_span(text, NER_AMOUNT_PATTERN)]
        + [(s, e, "DATE") for s, e in _find_span(text, NER_DATE_NUMERIC_PATTERN)]
        + [(s, e, "DATE") for s, e in _find_span(text, NER_DATE_TEXT_PATTERN)]
    )


def evaluate_regex_baseline(nlp, sub_df: pd.DataFrame) -> dict:
    examples = []
    for _, row in sub_df.iterrows():
        gold_doc = nlp.make_doc(row[CFG.text_col])
        gold_ents = []
        for start, end, label in row["ner_spans"]:
            span = gold_doc.char_span(start, end, label=label, alignment_mode="contract")
            if span is not None:
                gold_ents.append(span)
        gold_doc.ents = filter_spans(gold_ents)

        pred_doc = nlp.make_doc(row[CFG.text_col])
        pred_ents = []
        for start, end, label in regex_only_predict(row[CFG.text_col]):
            span = pred_doc.char_span(start, end, label=label, alignment_mode="contract")
            if span is not None:
                pred_ents.append(span)
        pred_doc.ents = filter_spans(pred_ents)

        examples.append(Example(pred_doc, gold_doc))
    return Scorer().score(examples)


regex_baseline_scores = evaluate_regex_baseline(nlp_ner, ner_test_df)
print("=== Baseline regex seule (sans VILLE, sans apprentissage) ===")
print(f"Precision : {regex_baseline_scores['ents_p']:.3f} | "
      f"Recall : {regex_baseline_scores['ents_r']:.3f} | "
      f"F1 : {regex_baseline_scores['ents_f']:.3f}")
print(f"\n(A comparer au F1 du modele spaCy = {ner_scores['ents_f']:.3f}, section 16.6)")
print("-> Si l'ecart est faible, la valeur ajoutee du ML se concentre presque entierement sur")
print("   VILLE (seul type non couvert par le regex) : un pipeline hybride regex+ML cible sur")
print("   VILLE serait alors plus simple et moins couteux qu'un modele spaCy entraine sur les 4 types.")


### 16.7 Analyse des erreurs NER

In [ ]:
def show_ner_errors(nlp, sub_df: pd.DataFrame, n=10):
    """Affiche les exemples ou les entites predites different des entites attendues (extraites
    via la pre-annotation + audit, section 16.2/16.3)."""
    shown = 0
    for _, row in sub_df.iterrows():
        pred_doc = nlp(row[CFG.text_col])
        pred_ents = {(e.start_char, e.end_char, e.label_) for e in pred_doc.ents}
        gold_ents = {(s, e, l) for s, e, l in row["ner_spans"]}
        if pred_ents != gold_ents:
            print(f"TEXTE : {row[CFG.text_col][:200]}")
            print(f"  attendu : {sorted(gold_ents)}")
            print(f"  predit  : {sorted(pred_ents)}")
            print()
            shown += 1
        if shown >= n:
            break

show_ner_errors(nlp_ner, ner_test_df, n=10)


## 17. Scoring de priorité des réclamations

Objectif : attribuer à chaque réclamation un niveau de priorité de traitement (`HAUTE` / `MOYENNE` / `BASSE`) pour orienter la file de traitement humain. Approche **hybride** :

- **Règles métier explicites** (rapides, interprétables, auditable) sur le texte brut : retard critique, dommage majeur, plainte forte / menace d'escalade, ton agressif.
- **Signal du modèle de classification** (section 10/11) : la catégorie prédite et la confiance associée, pour pondérer les cas où le texte seul est ambigu.

Les règles "dures" (dommage majeur, retard très long, plainte avec menace explicite) déclenchent directement `HAUTE`, sans dépendre du modèle -- un signal de sécurité ou d'escalade ne doit jamais être noyé dans un score continu. Le reste des cas passe par un score continu combinant signaux faibles + catégorie/confiance du modèle.

### 17.1 Règles métier de priorité

| Signal | Détection | Effet |
|---|---|---|
| Dommage majeur / sécurité | mots-clés (explosion, brûlure, danger, incendie...) | **HAUTE directe** |
| Retard très long | durée extraite du texte ≥ seuil urgent (défaut 14 j) | **HAUTE directe** |
| Plainte forte / escalade | mots-clés (avocat, tribunal, presse, résiliation, association de consommateurs...) | **HAUTE directe** |
| Perte / vol confirmé | mots-clés (volé, disparu, jamais reçu, introuvable...) | **HAUTE directe** |
| Dommage mineur | mots-clés (cassé, endommagé, défectueux...) | signal faible (+score) |
| Retard modéré | durée extraite ≥ seuil modéré (défaut 3 j), < seuil urgent | signal faible (+score) |
| Montant élevé mentionné | regex montant (€/EUR) ≥ seuil (défaut 300 €) | signal faible (+score) |
| Ton agressif | ratio de majuscules élevé et/ou ponctuation d'exclamation répétée | signal faible (+score) |
| Erreur de destination | mots-clés (mauvaise adresse, mauvais destinataire...) | signal faible (+score) |
| Catégorie x confiance modèle | sévérité par catégorie (`PRIORITY_CATEGORY_SEVERITY`) pondérée par la confiance de prédiction | signal modèle (+score) |

Les seuils et poids sont des constantes ajustables ci-dessous -- à recalibrer avec le métier une fois des cas réels observés (section 17.4).

In [ ]:
# --- Detection de signaux textuels pour le scoring de priorite ---
# Section auto-suffisante (patterns redefinis localement, independants de la section 16 NER)
# pour que score_priority() reste facilement extractible/reutilisable telle quelle.

RETARD_DUREE_PATTERN = re.compile(
    r"(\d{1,3})\s*(jour|jours|semaine|semaines|mois)\s+(?:de\s+)?retard", re.IGNORECASE
)
RETARD_DEPUIS_PATTERN = re.compile(
    r"depuis\s+(\d{1,3})\s*(jour|jours|semaine|semaines|mois)", re.IGNORECASE
)

DOMMAGE_MAJEUR_KEYWORDS = (
    "explos", "brul", "brûl", "bless", "danger", "incendie", "electrocut", "électrocut",
    "intoxicat", "grave", "hopital", "hôpital", "urgence vitale",
)
DOMMAGE_MINEUR_KEYWORDS = (
    "casse", "cassé", "cassée", "endommage", "endommagé", "abime", "abîmé",
    "defectueux", "défectueux", "raye", "rayé", "fuite", "tache",
)
PERTE_VOL_KEYWORDS = (
    "vole", "volé", "volee", "volée", "cambriol", "disparu", "disparition",
    "jamais recu", "jamais reçu", "jamais livre", "jamais livré", "introuvable",
    "colis perdu", "colis egare", "colis égaré", "envole", "envolé",
)
ERREUR_DESTINATION_KEYWORDS = (
    "mauvaise adresse", "erreur d'adresse", "erreur de destinataire",
    "livre a la mauvaise", "livré à la mauvaise", "mauvais destinataire",
    "chez quelqu'un d'autre", "faux destinataire", "adresse erronee", "adresse erronée",
)
ESCALADE_KEYWORDS = (
    "avocat", "juridique", "tribunal", "huissier", "presse", "media", "média",
    "reseaux sociaux", "réseaux sociaux", "association de consommateurs",
    "signalement", "resiliation", "résiliation", "derniere fois", "dernière fois",
    "inacceptable", "scandaleux", "remboursement immediat", "remboursement immédiat",
    "porter plainte", "plainte officielle",
)

MONTANT_PATTERN_PRIORITE = re.compile(
    r"(\d{1,3}(?:[ .]\d{3})*(?:[.,]\d{1,2})?|\d+(?:[.,]\d{1,2})?)\s?(?:€|eur\b|euros?\b)",
    re.IGNORECASE,
)


def _duree_en_jours(nombre: int, unite: str) -> int:
    """Convertit une duree extraite (nombre + unite textuelle) en nombre de jours."""
    unite = unite.lower()
    if unite.startswith("sem"):
        return nombre * 7
    if unite.startswith("mois"):
        return nombre * 30
    return nombre


def detecter_retard_jours(text: str) -> int | None:
    """Duree de retard la plus longue mentionnee dans le texte (en jours), ou None si
    aucune duree de retard n'est detectee."""
    durees = []
    for pattern in (RETARD_DUREE_PATTERN, RETARD_DEPUIS_PATTERN):
        for match in pattern.finditer(text):
            durees.append(_duree_en_jours(int(match.group(1)), match.group(2)))
    return max(durees) if durees else None


def detecter_dommage(text: str) -> tuple[bool, bool]:
    """Retourne (dommage_majeur, dommage_mineur) detectes dans le texte (recherche simple
    de sous-chaines, insensible a la casse -- suffisant pour un premier niveau de regle)."""
    text_lower = text.lower()
    majeur = any(kw in text_lower for kw in DOMMAGE_MAJEUR_KEYWORDS)
    mineur = any(kw in text_lower for kw in DOMMAGE_MINEUR_KEYWORDS)
    return majeur, mineur


def detecter_perte_vol(text: str) -> bool:
    """Detecte une perte ou un vol confirme (colis jamais recu / disparu / vole). Traite
    comme une regle dure au meme titre qu'un dommage majeur : le client n'a tout simplement
    rien recu, ce qui justifie une priorite HAUTE independamment de la categorie predite."""
    text_lower = text.lower()
    return any(kw in text_lower for kw in PERTE_VOL_KEYWORDS)


def detecter_erreur_destination(text: str) -> bool:
    """Detecte une mention explicite d'erreur d'adresse/destinataire. Signal faible : une
    erreur de destination genere un delai supplementaire mais reste en general recuperable
    (redirection, renvoi), donc pas une regle dure a elle seule."""
    text_lower = text.lower()
    return any(kw in text_lower for kw in ERREUR_DESTINATION_KEYWORDS)


def detecter_escalade(text: str) -> bool:
    """Detecte une menace d'escalade explicite (juridique, mediatique, associative...)."""
    text_lower = text.lower()
    return any(kw in text_lower for kw in ESCALADE_KEYWORDS)


def extraire_montant_eur(text: str) -> float | None:
    """Plus grand montant en euros mentionne dans le texte, ou None si aucun montant.
    Gere les separateurs de milliers (espace/point) et le separateur decimal (virgule ou
    point) usuels en francais : '1 200,50 EUR', '1.200,50€', '1200,5 euros', '85 euros'."""
    montants = []
    for match in MONTANT_PATTERN_PRIORITE.finditer(text):
        brut = match.group(1).strip().replace(" ", "")
        if "," in brut and "." in brut:
            brut = brut.replace(".", "").replace(",", ".")  # '.' = milliers, ',' = decimal
        elif "," in brut:
            brut = brut.replace(",", ".")  # ',' = decimal
        try:
            montants.append(float(brut))
        except ValueError:
            continue
    return max(montants) if montants else None


def detecter_ton_agressif(text: str, min_longueur: int = 15) -> bool:
    """Ton agressif = forte proportion de majuscules sur un texte assez long pour etre
    significatif, OU au moins 2 points d'exclamation."""
    lettres = [c for c in text if c.isalpha()]
    if len(lettres) >= min_longueur:
        ratio_maj = sum(1 for c in lettres if c.isupper()) / len(lettres)
        if ratio_maj > 0.3:
            return True
    return text.count("!") >= 2


### 17.2 Scoring hybride : `score_priority(text, category)`

`PRIORITY_CATEGORY_SEVERITY` associe à chaque catégorie (`categorie_finale`) une sévérité de base entre 0 et 1 -- initialisée automatiquement à partir des catégories réellement présentes dans les données (`classes`, section 8) avec une valeur neutre par défaut. **À ajuster avec le métier** une fois les vraies catégories connues, par ex. :

```python
PRIORITY_CATEGORY_SEVERITY["Produit dangereux"] = 0.9
PRIORITY_CATEGORY_SEVERITY["Demande d'information"] = 0.1
```

In [ ]:
# --- Configuration du scoring de priorite (ajustable) ---
RETARD_JOURS_SEUIL_URGENT = 14   # retard >= 14j -> HAUTE directe (regle dure)
RETARD_JOURS_SEUIL_MODERE = 3    # retard >= 3j (mais < seuil urgent) -> signal faible
MONTANT_SEUIL_ELEVE = 300.0      # montant mentionne >= 300 EUR -> signal faible

# Poids regles > modele : le texte de la reclamation reste la source de verite premiere ;
# le modele affine/nuance plutot que de dominer le score (coherent avec les regles "dures"
# ci-dessus qui, elles, ignorent completement le modele).
PRIORITY_WEIGHTS = {"regles": 0.8, "modele": 0.2}
PRIORITY_THRESHOLDS = {"haute": 0.55, "moyenne": 0.20}
DEFAULT_CATEGORY_SEVERITY = 0.45  # categorie inconnue -> severite neutre

# Severite de base par categorie -- calibree pour les 6 categories reelles du dataset
# (reclamations transport). Justification :
#   PERTE_VOL             (0.85) : colis jamais recu / vole -- quasi aussi grave qu'un
#                                  dommage majeur, mais couvert en plus par une regle dure
#                                  dediee (voir detecter_perte_vol ci-dessous).
#   PRODUIT_ENDOMMAGE      (0.70) : marchandise recue en mauvais etat -- prejudice concret,
#                                  rarement un enjeu de securite (sinon capte par la regle
#                                  dure dommage_majeur independamment de la categorie).
#   ERREUR_DESTINATION     (0.60) : colis livre au mauvais endroit -- delai supplementaire
#                                  et risque de perte si non recupere.
#   RETOUR_REMBOURSEMENT   (0.55) : enjeu financier direct pour le client, mais rarement
#                                  urgent en soi (traite par les process retour standard).
#   QUALITE_SERVICE        (0.35) : insatisfaction sur le service (communication, delais
#                                  percus, amabilite...) -- gene reelle mais rarement
#                                  bloquante pour le client.
#   AUTRE                  (0.30) : categorie fourre-tout, traitee au fil de l'eau.
# A recalibrer avec le metier des que des retours terrain sont disponibles (section 17.4).
PRIORITY_CATEGORY_SEVERITY: dict = {
    "PERTE_VOL": 0.85,
    "PRODUIT_ENDOMMAGE": 0.70,
    "ERREUR_DESTINATION": 0.60,
    "RETOUR_REMBOURSEMENT": 0.55,
    "QUALITE_SERVICE": 0.35,
    "AUTRE": 0.30,
}
# Filet de securite : toute categorie du dataset non listee explicitement ci-dessus recoit
# la severite neutre par defaut plutot que de faire planter le scoring.
if "classes" in globals():
    for _c in classes:
        PRIORITY_CATEGORY_SEVERITY.setdefault(_c, DEFAULT_CATEGORY_SEVERITY)


def score_priority(text: str, category: str, model_confidence: float | None = None) -> dict:
    """Calcule la priorite de traitement d'une reclamation.

    Hybride : des regles metier "dures" (dommage majeur, retard tres long, escalade
    explicite) forcent directement HAUTE. En dehors de ces cas, un score continu combine
    des signaux textuels faibles (regles metier "souples") et un signal modele
    (severite de la categorie ponderee par la confiance de prediction).

    Args:
        text: texte brut de la reclamation.
        category: categorie (predite ou reelle) de la reclamation.
        model_confidence: confiance du modele de classification dans `category`
            (probabilite entre 0 et 1). Si None, le score du modele est neutre (0.5) sauf
            si un pipeline entraine `best_baseline_model` est disponible dans
            l'environnement, auquel cas la confiance est calculee automatiquement.

    Returns:
        dict avec les cles "priorite" (HAUTE/MOYENNE/BASSE), "score" (float 0-1),
        "declencheur_regle_dure" (str ou None) et "signaux" (detail des detections,
        pour tracabilite/audit).
    """
    text = text if isinstance(text, str) else ""

    # --- Confiance modele : auto-calcul si non fournie et pipeline disponible ---
    if model_confidence is None and text.strip() and "best_baseline_model" in globals():
        try:
            text_clean = clean_text_classic(text) if "clean_text_classic" in globals() else text
            proba_row = best_baseline_model.predict_proba([text_clean])[0]
            model_classes = list(best_baseline_model.named_steps["clf"].classes_)
            if category in model_classes:
                model_confidence = float(proba_row[model_classes.index(category)])
        except Exception:
            model_confidence = None  # pipeline absent/incompatible -> repli neutre

    # --- Detection des signaux textuels ---
    retard_jours = detecter_retard_jours(text)
    dommage_majeur, dommage_mineur = detecter_dommage(text)
    escalade = detecter_escalade(text)
    perte_vol = detecter_perte_vol(text)
    erreur_destination = detecter_erreur_destination(text)
    montant_eur = extraire_montant_eur(text)
    ton_agressif = detecter_ton_agressif(text)
    categorie_severite = PRIORITY_CATEGORY_SEVERITY.get(category, DEFAULT_CATEGORY_SEVERITY)

    signaux = {
        "retard_jours_detecte": retard_jours,
        "dommage_majeur_detecte": dommage_majeur,
        "dommage_mineur_detecte": dommage_mineur,
        "plainte_forte_detectee": escalade,
        "perte_vol_detectee": perte_vol,
        "erreur_destination_detectee": erreur_destination,
        "montant_detecte_eur": montant_eur,
        "ton_agressif": ton_agressif,
        "categorie_severite": categorie_severite,
        "confiance_modele": model_confidence,
    }

    # --- Regles dures : HAUTE immediate, sans dependre du score continu ---
    if dommage_majeur:
        return {"priorite": "HAUTE", "score": 1.0, "declencheur_regle_dure": "dommage_majeur", "signaux": signaux}
    if retard_jours is not None and retard_jours >= RETARD_JOURS_SEUIL_URGENT:
        return {"priorite": "HAUTE", "score": 1.0, "declencheur_regle_dure": "retard_urgent", "signaux": signaux}
    if escalade:
        return {"priorite": "HAUTE", "score": 1.0, "declencheur_regle_dure": "plainte_forte", "signaux": signaux}
    if perte_vol:
        return {"priorite": "HAUTE", "score": 1.0, "declencheur_regle_dure": "perte_vol_confirme", "signaux": signaux}

    # --- Score continu (regles souples + signal modele) ---
    points_regles = 0.0
    if dommage_mineur:
        points_regles += 0.25
    if retard_jours is not None and retard_jours >= RETARD_JOURS_SEUIL_MODERE:
        points_regles += 0.25
    if montant_eur is not None and montant_eur >= MONTANT_SEUIL_ELEVE:
        points_regles += 0.25
    if ton_agressif:
        points_regles += 0.15
    if erreur_destination:
        points_regles += 0.20
    rule_score = min(points_regles, 1.0)

    confiance_pour_score = model_confidence if model_confidence is not None else 0.5
    model_signal = categorie_severite * confiance_pour_score

    score = PRIORITY_WEIGHTS["regles"] * rule_score + PRIORITY_WEIGHTS["modele"] * model_signal
    score = round(min(max(score, 0.0), 1.0), 4)

    if score >= PRIORITY_THRESHOLDS["haute"]:
        priorite = "HAUTE"
    elif score >= PRIORITY_THRESHOLDS["moyenne"]:
        priorite = "MOYENNE"
    else:
        priorite = "BASSE"

    return {"priorite": priorite, "score": score, "declencheur_regle_dure": None, "signaux": signaux}


# --- Demonstration rapide (categories reelles du dataset transport) ---
exemple_1 = score_priority(
    "Mon colis est marque comme livre mais je ne l'ai jamais recu, je pense qu'il a ete vole.",
    "PERTE_VOL",
)
exemple_2 = score_priority(
    "Le carton est arrive ecrase et le produit a l'interieur est casse.",
    "PRODUIT_ENDOMMAGE", model_confidence=0.8,
)
for _label, _ex in [("perte/vol", exemple_1), ("produit endommage", exemple_2)]:
    print(f"--- {_label} ---")
    print(json.dumps(_ex, indent=2, ensure_ascii=False))


### 17.3 Tests unitaires

Tests des règles dures, du score continu et des fonctions de détection unitaires. `model_confidence` est passé explicitement dans la plupart des cas pour rendre les tests déterministes et indépendants d'un modèle entraîné.

In [ ]:
import unittest


class TestDetecteurs(unittest.TestCase):
    """Tests unitaires des fonctions de detection de signaux."""

    def test_retard_avec_mention_explicite(self):
        self.assertEqual(detecter_retard_jours("Ma commande a 10 jours de retard."), 10)

    def test_retard_avec_depuis_en_semaines(self):
        self.assertEqual(detecter_retard_jours("Toujours rien recu depuis 3 semaines."), 21)

    def test_retard_absent(self):
        self.assertIsNone(detecter_retard_jours("Le produit correspond a la description."))

    def test_retard_prend_la_duree_maximale(self):
        self.assertEqual(
            detecter_retard_jours("3 jours de retard sur ce colis, mais depuis 2 mois sur l'autre."),
            60,
        )

    def test_dommage_majeur_detecte(self):
        majeur, mineur = detecter_dommage("Le chargeur a explose et m'a brule la main.")
        self.assertTrue(majeur)

    def test_dommage_mineur_detecte(self):
        majeur, mineur = detecter_dommage("Le produit est arrive casse.")
        self.assertFalse(majeur)
        self.assertTrue(mineur)

    def test_aucun_dommage(self):
        majeur, mineur = detecter_dommage("Bonjour, je voudrais des informations.")
        self.assertFalse(majeur)
        self.assertFalse(mineur)

    def test_escalade_detectee(self):
        self.assertTrue(detecter_escalade("Je vais saisir mon avocat si rien ne change."))

    def test_escalade_absente(self):
        self.assertFalse(detecter_escalade("Merci de votre reponse rapide."))

    def test_montant_simple(self):
        self.assertAlmostEqual(extraire_montant_eur("Facture de 85 euros recue."), 85.0)

    def test_montant_avec_symbole(self):
        self.assertAlmostEqual(extraire_montant_eur("Paye 1200€ pour rien."), 1200.0)

    def test_montant_absent(self):
        self.assertIsNone(extraire_montant_eur("Pas de montant ici."))

    def test_ton_agressif_majuscules(self):
        self.assertTrue(detecter_ton_agressif("C'EST INADMISSIBLE ET INACCEPTABLE"))

    def test_ton_agressif_exclamations(self):
        self.assertTrue(detecter_ton_agressif("Ca ne va pas du tout !! Repondez vite !!"))

    def test_ton_neutre(self):
        self.assertFalse(detecter_ton_agressif("Bonjour, pourriez-vous verifier ma commande ?"))


class TestScorePriority(unittest.TestCase):
    """Tests unitaires de la fonction hybride score_priority."""

    def test_haute_dommage_majeur(self):
        r = score_priority(
            "Le produit a explose et m'a brule la main, tres dangereux !!!",
            "PRODUIT_ENDOMMAGE", model_confidence=0.9,
        )
        self.assertEqual(r["priorite"], "HAUTE")
        self.assertEqual(r["declencheur_regle_dure"], "dommage_majeur")

    def test_haute_retard_urgent(self):
        r = score_priority(
            "Toujours pas livre depuis 3 semaines, ou est mon colis ?",
            "ERREUR_DESTINATION", model_confidence=0.8,
        )
        self.assertEqual(r["priorite"], "HAUTE")
        self.assertEqual(r["declencheur_regle_dure"], "retard_urgent")

    def test_haute_plainte_forte(self):
        r = score_priority(
            "Remboursez-moi immediatement ou je saisis une association de consommateurs.",
            "RETOUR_REMBOURSEMENT", model_confidence=0.6,
        )
        self.assertEqual(r["priorite"], "HAUTE")
        self.assertEqual(r["declencheur_regle_dure"], "plainte_forte")

    def test_basse_demande_info(self):
        r = score_priority(
            "Bonjour, je voudrais juste savoir comment suivre ma commande, merci.",
            "QUALITE_SERVICE", model_confidence=0.9,
        )
        self.assertEqual(r["priorite"], "BASSE")
        self.assertIsNone(r["declencheur_regle_dure"])

    def test_haute_perte_vol_confirmee(self):
        r = score_priority(
            "Mon colis n'est jamais arrive, le suivi indique livre mais je pense qu'il a ete vole.",
            "PERTE_VOL", model_confidence=0.85,
        )
        self.assertEqual(r["priorite"], "HAUTE")
        self.assertEqual(r["declencheur_regle_dure"], "perte_vol_confirme")

    def test_signal_erreur_destination(self):
        r = score_priority(
            "Mon colis a ete livre a la mauvaise adresse, chez quelqu'un d'autre.",
            "ERREUR_DESTINATION", model_confidence=0.7,
        )
        self.assertTrue(r["signaux"]["erreur_destination_detectee"])
        self.assertIsNone(r["declencheur_regle_dure"])

    def test_severite_categories_reelles(self):
        """A signaux et confiance identiques, PERTE_VOL doit toujours scorer plus haut que
        AUTRE : verifie que le classement des severites de categories est respecte."""
        texte = "Reclamation standard sans signal textuel particulier."
        r_perte_vol = score_priority(texte, "PERTE_VOL", model_confidence=0.7)
        r_autre = score_priority(texte, "AUTRE", model_confidence=0.7)
        self.assertGreater(r_perte_vol["score"], r_autre["score"])

    def test_moyenne_retard_modere(self):
        r = score_priority(
            "Ma commande a 4 jours de retard, pouvez-vous me dire ou elle en est ?",
            "ERREUR_DESTINATION", model_confidence=0.5,
        )
        self.assertIn(r["priorite"], ("MOYENNE", "HAUTE"))
        self.assertIsNone(r["declencheur_regle_dure"])

    def test_score_croissant_avec_confiance_modele(self):
        """A signaux textuels egaux, une confiance modele plus elevee sur une categorie
        severe ne doit jamais faire BAISSER le score."""
        PRIORITY_CATEGORY_SEVERITY["__test_categorie_severe__"] = 0.9
        texte = "Le produit ne fonctionne pas comme prevu."
        r_basse_confiance = score_priority(texte, "__test_categorie_severe__", model_confidence=0.1)
        r_haute_confiance = score_priority(texte, "__test_categorie_severe__", model_confidence=0.95)
        self.assertGreaterEqual(r_haute_confiance["score"], r_basse_confiance["score"])
        del PRIORITY_CATEGORY_SEVERITY["__test_categorie_severe__"]

    def test_categorie_inconnue_utilise_severite_par_defaut(self):
        r = score_priority("Texte neutre sans signal particulier.", "__categorie_jamais_vue__", model_confidence=0.5)
        self.assertEqual(r["signaux"]["categorie_severite"], DEFAULT_CATEGORY_SEVERITY)

    def test_texte_vide(self):
        r = score_priority("", "AUTRE", model_confidence=0.5)
        self.assertEqual(r["priorite"], "BASSE")
        self.assertIsNone(r["declencheur_regle_dure"])

    def test_texte_none_ne_plante_pas(self):
        r = score_priority(None, "AUTRE", model_confidence=0.5)
        self.assertEqual(r["priorite"], "BASSE")

    def test_sans_confiance_modele_fournie_score_neutre(self):
        """Sans best_baseline_model dans l'environnement ni model_confidence fourni,
        le signal modele doit utiliser le repli neutre (0.5), pas planter."""
        _backup = globals().pop("best_baseline_model", "__absent__")
        try:
            r = score_priority("Produit correct, rien a signaler.", "QUALITE_SERVICE")
            self.assertIsNone(r["signaux"]["confiance_modele"])
        finally:
            if _backup != "__absent__":
                globals()["best_baseline_model"] = _backup


_runner = unittest.TextTestRunner(verbosity=2)
_suite = unittest.TestLoader().loadTestsFromModule(__import__("__main__"))
_result = _runner.run(_suite)
assert _result.wasSuccessful(), "Des tests unitaires de scoring de priorite ont echoue."


### 17.4 Test sur des cas réels (test set)

On applique `score_priority` aux réclamations du `test_df`, en utilisant la catégorie **prédite** par la baseline LR et sa confiance associée (déjà calculées en section 14.1) comme signal modèle -- c'est le scénario de production réel : au moment de scorer une nouvelle réclamation, on n'a que la prédiction du modèle, pas la vraie catégorie.

In [ ]:
# --- Application sur le test set : categorie predite + confiance predite (deja calculees en 14.1) ---
priority_test_df = test_df.copy().reset_index(drop=True)
priority_test_df["categorie_predite"] = y_test_pred_lr
priority_test_df["confiance_predite"] = lr_proba.max(axis=1)

priority_results = priority_test_df.apply(
    lambda r: score_priority(r[CFG.text_col], r["categorie_predite"], model_confidence=r["confiance_predite"]),
    axis=1,
)
priority_test_df["priorite"] = priority_results.apply(lambda d: d["priorite"])
priority_test_df["score_priorite"] = priority_results.apply(lambda d: d["score"])
priority_test_df["declencheur_regle_dure"] = priority_results.apply(lambda d: d["declencheur_regle_dure"])

print("Repartition des priorites sur le test set :")
display(priority_test_df["priorite"].value_counts().rename("nombre"))

print("\nRepartition des declencheurs de regle dure (priorite HAUTE) :")
display(priority_test_df.loc[priority_test_df["priorite"] == "HAUTE", "declencheur_regle_dure"].value_counts())

print("\nTop 10 reclamations les plus prioritaires :")
top_priorite = priority_test_df.sort_values("score_priorite", ascending=False).head(10)
for _, row in top_priorite.iterrows():
    print(f"[{row['priorite']:7s} | score={row['score_priorite']:.2f} | categorie={row['categorie_predite']}]")
    print(f"  {row[CFG.text_col]}\n")

priority_report_path = CFG.reports_dir / "scoring_priorite_test_set.csv"
priority_test_df[[CFG.text_col, "categorie_predite", "confiance_predite", "priorite", "score_priorite", "declencheur_regle_dure"]].to_csv(
    priority_report_path, index=False
)
logger.info("Scoring de priorite sur le test set sauvegarde dans %s", priority_report_path)


## 18. Orchestration bout en bout : `traiter_reclamation(text)`

Assemble les trois briques précédentes (classification, NER, scoring de priorité) en une seule fonction qui prend une réclamation brute et retourne un résultat consolidé, avec **routage automatique vers une équipe** selon la catégorie prédite.

> **Remarque** : `RETARD` n'existe pas dans les 6 catégories réelles du dataset (`RETOUR_REMBOURSEMENT`, `PERTE_VOL`, `QUALITE_SERVICE`, `ERREUR_DESTINATION`, `AUTRE`, `PRODUIT_ENDOMMAGE`) -- le modèle ne peut prédire que parmi celles-ci. Un retard pur (sans erreur d'adresse, perte ni dommage) atterrit aujourd'hui plutôt dans `QUALITE_SERVICE` ou `AUTRE`, mais reste détecté comme **signal de priorité** via `detecter_retard_jours` (section 17.1). Si le retard doit devenir une catégorie à part entière, il faut ré-annoter une partie du dataset avec ce label avant de ré-entraîner (section 10/11).

### 18.0 Rechargement rapide (sans ré-entraînement)

Pour tester `traiter_reclamation` sur de nouvelles réclamations sans relancer tout le pipeline (nettoyage, entraînement LR, entraînement NER...), il suffit d'exécuter, dans l'ordre :
1. Les imports de base (section 2, cellules 1-2) -- pas besoin de charger les données ni d'entraîner quoi que ce soit.
2. **Cette cellule** (18.0) -- recharge les modèles déjà entraînés depuis le disque.
3. Les cellules de fonctions des sections 17 (`score_priority` et ses dépendances) et 18.1/18.2/18.3.

⚠️ **Sur Colab, `/content` est vidé si le runtime redémarre complètement** (pas juste "Exécuter tout", mais une vraie déconnexion/reconnexion). Dans ce cas, avant d'exécuter cette cellule, il faut soit monter Google Drive et y avoir copié le dossier `models/`, soit re-uploader et décompresser le zip téléchargé en section 15 dans `/content/models`.

In [ ]:
# --- Rechargement rapide des modeles deja entraines (sections 10 et 16.5) ---
import re
import unicodedata
import joblib
import spacy
from pathlib import Path

MODELS_DIR = Path("/content/models")  # doit correspondre a CFG.models_dir (section 2)

# Fonction de nettoyage necessaire a la classification (identique a la section 5),
# redefinie ici pour ne pas dependre de l'execution complete du notebook.
_TRACKING_PATTERN = re.compile(r"\bTRK-\w+\b", flags=re.IGNORECASE)
_MULTI_SPACE_PATTERN = re.compile(r"\s+")
_CENSORSHIP_PATTERN = re.compile(r"\*{2,}")


def _base_clean(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text))
    text = _TRACKING_PATTERN.sub("__TRACKING__", text)
    text = _CENSORSHIP_PATTERN.sub(" __CENSURE__ ", text)
    text = _MULTI_SPACE_PATTERN.sub(" ", text).strip()
    return text


def clean_text_classic(text: str) -> str:
    text = _base_clean(text)
    text = text.lower()
    text = re.sub(r"[^\w\s_]", " ", text, flags=re.UNICODE)
    text = _MULTI_SPACE_PATTERN.sub(" ", text).strip()
    return text


# --- Chargement des modeles entraines ---
_baseline_path = MODELS_DIR / "baseline_tfidf_logreg_pipeline.joblib"
_ner_path = MODELS_DIR / "ner" / "model-best"

if not _baseline_path.exists() or not _ner_path.exists():
    raise FileNotFoundError(
        f"Modeles introuvables ({_baseline_path} / {_ner_path}). "
        "Si le runtime Colab a redemarre, remontez Google Drive ou re-uploadez le zip "
        "de la section 15 dans /content/models avant de relancer cette cellule."
    )

best_baseline_model = joblib.load(_baseline_path)
nlp_ner = spacy.load(_ner_path)

print("Modeles recharges avec succes :")
print(f"  Baseline (TF-IDF + LR) : {_baseline_path}")
print(f"  NER (spaCy)            : {_ner_path}")


### 18.1 Routage vers une équipe

Table de correspondance catégorie → équipe, ajustable avec le métier. Un déclencheur de règle dure "grave" (dommage majeur, plainte forte) ajoute en plus un signal d'escalade manager, indépendamment de l'équipe cible normale.

In [ ]:
# --- Table de routage : categorie -> equipe en charge du traitement ---
ROUTING_TEAM = {
    "PERTE_VOL": "Logistique",
    "ERREUR_DESTINATION": "Logistique",
    "PRODUIT_ENDOMMAGE": "Qualité / SAV",
    "RETOUR_REMBOURSEMENT": "Finance / Remboursements",
    "QUALITE_SERVICE": "Service Après-Vente",
    "AUTRE": "Support général",
}
DEFAULT_TEAM = "Support général"

# Declencheurs de regle dure qui justifient une alerte manager en plus du routage normal
# (perte_vol_confirme est deja gere par l'equipe Logistique elle-meme, pas d'escalade requise).
ESCALADE_DECLENCHEURS = {"dommage_majeur", "plainte_forte"}


def router_equipe(categorie: str, declencheur_regle_dure: str | None = None) -> dict:
    """Determine l'equipe cible et si une escalade manager est necessaire.

    Args:
        categorie: categorie predite de la reclamation.
        declencheur_regle_dure: cle du declencheur de regle dure (voir score_priority),
            ou None si le score continu a ete utilise.

    Returns:
        dict avec "equipe" (str) et "escalade_manager" (bool).
    """
    return {
        "equipe": ROUTING_TEAM.get(categorie, DEFAULT_TEAM),
        "escalade_manager": declencheur_regle_dure in ESCALADE_DECLENCHEURS,
    }


### 18.2 Fonction d'orchestration

`traiter_reclamation` enchaîne : nettoyage + classification (section 10), extraction d'entités (section 16), scoring de priorité (section 17), puis routage (18.1).

In [ ]:
# Correspondance entre les labels NER (section 16.1) et des cles courtes et lisibles
ENTITY_LABEL_TO_KEY = {
    "NUMERO_TRACKING": "tracking",
    "VILLE": "ville",
    "MONTANT": "montant",
    "DATE": "date",
}


def extraire_entites_resume(text: str) -> dict:
    """Version condensee des entites NER : une valeur par type (la premiere rencontree),
    pour un affichage synthetique. Utiliser directement nlp_ner(text).ents si toutes les
    occurrences sont necessaires (ex. plusieurs montants dans un meme message)."""
    doc = nlp_ner(text)
    resume = {}
    for ent in doc.ents:
        cle = ENTITY_LABEL_TO_KEY.get(ent.label_, ent.label_.lower())
        resume.setdefault(cle, ent.text)
    return resume


def traiter_reclamation(text: str) -> dict:
    """Orchestration bout en bout d'une reclamation brute.

    Enchaine classification (baseline TF-IDF + LR), extraction d'entites (NER spaCy),
    scoring de priorite hybride et routage vers une equipe.

    Args:
        text: texte brut de la reclamation.

    Returns:
        dict consolide : reclamation_brute, categorie, confiance_categorie, entites,
        priorite, score_priorite, declencheur_regle_dure, equipe_routage, escalade_manager.
    """
    text_clean = clean_text_classic(text)
    categorie = str(best_baseline_model.predict([text_clean])[0])
    confiance = float(best_baseline_model.predict_proba([text_clean]).max())

    entites = extraire_entites_resume(text)

    priorite_result = score_priority(text, categorie, model_confidence=confiance)
    routage = router_equipe(categorie, priorite_result["declencheur_regle_dure"])

    return {
        "reclamation_brute": text,
        "categorie": categorie,
        "confiance_categorie": round(confiance, 3),
        "entites": entites,
        "priorite": priorite_result["priorite"],
        "score_priorite": priorite_result["score"],
        "declencheur_regle_dure": priorite_result["declencheur_regle_dure"],
        "equipe_routage": routage["equipe"],
        "escalade_manager": routage["escalade_manager"],
    }


### 18.3 Affichage et démonstration

In [ ]:
def afficher_traitement(resultat: dict, detail: bool = False) -> None:
    """Affiche le resultat de traiter_reclamation au format ticket, concis par defaut.

    Args:
        resultat: sortie de traiter_reclamation.
        detail: si True, affiche aussi la confiance du modele et le score de priorite
            (utile en debug/audit, mais pas dans le ticket standard).
    """
    print("Réclamation brute :")
    print(f'  "{resultat["reclamation_brute"]}"')
    print()
    print(f"Catégorie : {resultat['categorie']}")
    print(f"Entités : {resultat['entites']}")
    print(f"Priorité : {resultat['priorite']}")
    print(f"Routage automatique -> Équipe : {resultat['equipe_routage']}")
    if resultat["escalade_manager"]:
        print("Escalade -> notifier le manager (dommage majeur ou plainte forte)")
    if detail:
        print()
        print(f"[detail] confiance categorie : {resultat['confiance_categorie']:.0%}")
        print(f"[detail] score priorite      : {resultat['score_priorite']:.2f}")
        if resultat["declencheur_regle_dure"]:
            print(f"[detail] declencheur regle dure : {resultat['declencheur_regle_dure']}")


# --- Demonstration ---
exemple_reclamation = (
    "Mon colis TRK-2890 est arrive a la mauvaise adresse a Sfax, "
    "ca fait 5 jours que j'attends une solution."
)
resultat_demo = traiter_reclamation(exemple_reclamation)
afficher_traitement(resultat_demo)


### 18.4 Tests unitaires du routage

In [ ]:
class TestRoutage(unittest.TestCase):
    """Tests unitaires de router_equipe (independants du modele/NER entraines)."""

    def test_routage_perte_vol_vers_logistique(self):
        r = router_equipe("PERTE_VOL", declencheur_regle_dure="perte_vol_confirme")
        self.assertEqual(r["equipe"], "Logistique")
        self.assertFalse(r["escalade_manager"])

    def test_routage_dommage_declenche_escalade(self):
        r = router_equipe("PRODUIT_ENDOMMAGE", declencheur_regle_dure="dommage_majeur")
        self.assertEqual(r["equipe"], "Qualité / SAV")
        self.assertTrue(r["escalade_manager"])

    def test_routage_plainte_forte_declenche_escalade(self):
        r = router_equipe("QUALITE_SERVICE", declencheur_regle_dure="plainte_forte")
        self.assertTrue(r["escalade_manager"])

    def test_routage_categorie_inconnue_repli_support(self):
        r = router_equipe("__categorie_jamais_vue__")
        self.assertEqual(r["equipe"], DEFAULT_TEAM)

    def test_routage_sans_declencheur_pas_escalade(self):
        r = router_equipe("RETOUR_REMBOURSEMENT", declencheur_regle_dure=None)
        self.assertFalse(r["escalade_manager"])


_result_routage = unittest.TextTestRunner(verbosity=2).run(
    unittest.TestLoader().loadTestsFromTestCase(TestRoutage)
)
assert _result_routage.wasSuccessful(), "Des tests unitaires de routage ont echoue."
